In [ ]:
# !pip -q install import-ipynb

In [ ]:
'''
# اول PyTorch 2.6 با CUDA 12.6 و ماژول‌های وابسته
!python -m pip install --upgrade --extra-index-url https://download.pytorch.org/whl/cu126 \
  torch==2.6.0+cu126 torchvision==0.21.0+cu126 torchaudio==2.6.0+cu126

# بعد MONAI پایدار و بقیه ابزارها
!python -m pip install monai==1.5.0 nibabel pydicom tqdm scikit-image
'''

'\n# اول PyTorch 2.6 با CUDA 12.6 و ماژول\u200cهای وابسته\n!python -m pip install --upgrade --extra-index-url https://download.pytorch.org/whl/cu126   torch==2.6.0+cu126 torchvision==0.21.0+cu126 torchaudio==2.6.0+cu126\n\n# بعد MONAI پایدار و بقیه ابزارها\n!python -m pip install monai==1.5.0 nibabel pydicom tqdm scikit-image\n'

In [ ]:
'''
import torch, torchvision, torchaudio, monai
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)
print("monai:", monai.__version__)
'''

'\nimport torch, torchvision, torchaudio, monai\nprint("torch:", torch.__version__)\nprint("torchvision:", torchvision.__version__)\nprint("torchaudio:", torchaudio.__version__)\nprint("monai:", monai.__version__)\n'

In [ ]:
#!pip -q install rarfile
#!apt -y install -qq unrar

In [ ]:
# نصب وابستگی‌ها (MONAI مخصوص MRI، به‌همراه nibabel و pydicom)
#!pip -q install -U monai nibabel pydicom tqdm scikit-image

In [ ]:
#@title ✏️ preprocessing
#@markdown


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
'''
# برو داخل پوشه‌ای که نوت‌بوک‌ت اونجاست
%cd "/content/drive/MyDrive/Colab Notebooks"

# اگر دفعه‌ی قبل import شکست خورده، نسخه‌ی ناقصِ ماژول رو از کشِ پایتون پاک کن
import sys
sys.modules.pop("img_preprocessing", None)

# فعال‌سازی ایمپورت ipynb و ایمپورتِ نوت‌بوک
!pip -q install import-ipynb
import import_ipynb

import img_preprocessing
# یا اگر فقط کلاس/توابع لازم داری:
from img_preprocessing import ImageToImage3D_SmallScale, transform_train
from torch.utils.data import DataLoader
'''

'\n# برو داخل پوشه\u200cای که نوت\u200cبوک\u200cت اونجاست\n%cd "/content/drive/MyDrive/Colab Notebooks"\n\n# اگر دفعه\u200cی قبل import شکست خورده، نسخه\u200cی ناقصِ ماژول رو از کشِ پایتون پاک کن\nimport sys\nsys.modules.pop("img_preprocessing", None)\n\n# فعال\u200cسازی ایمپورت ipynb و ایمپورتِ نوت\u200cبوک\n!pip -q install import-ipynb\nimport import_ipynb\n\nimport img_preprocessing\n# یا اگر فقط کلاس/توابع لازم داری:\nfrom img_preprocessing import ImageToImage3D_SmallScale, transform_train\nfrom torch.utils.data import DataLoader\n'

In [ ]:
'''
modelname = "gated3d"
imgsize = 224
imgdepth = 160
i=1
kfold=5
batch_size = 1
path = "/content/drive/MyDrive/dataset"

train_dataset = ImageToImage3D_SmallScale(dataset_path=path, i =1 , training = True,
                                          shape = ( imgsize,imgsize , imgdepth),
                                          kfold = kfold, transform=transform_train,
                                          verbose=True
                                         )
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


batch = next(iter(dataloader))
_, _, _, _, image_aug, mask_aug, image_id = batch
'''

'\nmodelname = "gated3d"\nimgsize = 224\nimgdepth = 160\ni=1\nkfold=5\nbatch_size = 1\npath = "/content/drive/MyDrive/dataset"\n\ntrain_dataset = ImageToImage3D_SmallScale(dataset_path=path, i =1 , training = True,\n                                          shape = ( imgsize,imgsize , imgdepth),\n                                          kfold = kfold, transform=transform_train,\n                                          verbose=True\n                                         )\ndataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)\n\n\nbatch = next(iter(dataloader))\n_, _, _, _, image_aug, mask_aug, image_id = batch\n'

In [ ]:
'''
print(image_id)
print(image_aug.shape)
print(mask_aug.shape)
'''

'\nprint(image_id)\nprint(image_aug.shape)\nprint(mask_aug.shape)\n'

In [ ]:
# 💠 Encoder block

In [ ]:
'''
# هایپرها
BASE = 32
in_channels=1
return_all=True
'''

'\n# هایپرها\nBASE = 32\nin_channels=1\nreturn_all=True\n'

In [ ]:
'''
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device
'''

'\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\ndevice\n'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvStem(nn.Module):
    """
    Convolutional Stem برای هر مدالیته:
    - سه بلوک conv3d → IN → LeakyReLU
    - دو بار downsample با stride=2 (سطح‌های 1/2 و 1/4)
    - کانال‌ها برحسب channel_mult قابل تنظیم‌اند (برای MCCA پیش‌فرض = ثابت)
    """
    def __init__(
        self,
        in_channels: int = 1,
        base_channels: int = 32,
        return_all: bool = False,
        channel_mult=(1, 1, 1),
        # ← پیش‌فرض: هر سه سطح همان base
        # پارامتر channel_mult برای کنترل تعداد کانال‌ها در هر سطح (پیش‌فرض همون base نگه می‌داره تا با MCCA سازگار باشه)
        #اگر خواستی ظرفیت را در سطح‌های پایین‌تر زیاد کنی، فقط این پارامتر را عوض می‌کنی؛ در عین حال پیش‌فرض با MCCA (که انتظار dim=base) سازگار می‌ماند.

        norm: str = "instance" ,
        # "instance" یا "group"
        negative_slope: float = 0.1
    ):
        '''
        برای حل مشکل، LeakyReLU اینطوری تعریف میشه:
        اگر x>0           x
        اگر x<=0            negative_slope*x
        عنی برای بخش منفی، به جای صفر، یه شیب کوچیک نگه می‌داره.
        همون شیب بخش منفی هست
        '''
        super().__init__()
        self.return_all = return_all
        c0 = int(base_channels * channel_mult[0])
        c1 = int(base_channels * channel_mult[1])
        c2 = int(base_channels * channel_mult[2])

        self.conv1 = self._make_block(in_channels, c0, stride=1, norm=norm, negative_slope=negative_slope)
        self.conv2 = self._make_block(c0, c1, stride=2, norm=norm, negative_slope=negative_slope)
        self.conv3 = self._make_block(c1, c2, stride=2, norm=norm, negative_slope=negative_slope)

    @staticmethod
    def _norm_act(ch, norm: str, negative_slope: float):
        # InstanceNorm/GroupNorm
        # با Batch کوچک (1–2) نوسان BN را ندارند → همگرایی روان‌تر.
        if norm == "group":
            # گروه‌ها را محدود نگه دار تا از ch بزرگ‌تر نشود
            groups = min(8, ch)
            return nn.Sequential(
                nn.GroupNorm(groups, ch, affine=True),
                nn.LeakyReLU(negative_slope, inplace=True)
                # LeakyReLU (پایدارتر از ReLU در 3D)
            )
        else:
            return nn.Sequential(
                nn.InstanceNorm3d(ch, affine=True),
                nn.LeakyReLU(negative_slope, inplace=True)
            )

    def _make_block(self, in_ch, out_ch, stride, norm: str, negative_slope: float):
        return nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False),
            # bias=False وقتی بلافاصله Normalization داریم، اضافه‌کاری است.
            self._norm_act(out_ch, norm, negative_slope),
        )

    def forward(self, x):
        x0 = self.conv1(x)   # 1x
        x1 = self.conv2(x0)  # 1/2
        x2 = self.conv3(x1)  # 1/4
        return (x0, x1, x2) if self.return_all else x2

    @property
    # دکوراتور @property باعث میشه که این متد مثل یک attribute استفاده بشه، نه مثل یک تابع.
    def out_channels(self):
        ''' بقیه‌ی ماژول‌ها بدون اینکه وارد جزئیات داخلی ConvStem بشن، بتونن بپرسن:
         «آخرین لایه‌ی کانولوشنی تو چند تا کانال خروجی داره؟»
        وقتی ماژول‌های بعدی (مثل MCCA یا bottleneck) می‌خوان بدونن خروجی ConvStem چند کانال داره، لازم نیست بیان مستقیم تو کد ConvStem جستجو کنن.
        این رویکرد باعث میشه کلاس encapsulation خوبی داشته باشه و تغییر ساختار داخلی ConvStem روی کد بقیه‌ی ماژول‌ها تاثیر نذاره.
        '''
        # برای اطلاع به ماژول‌های بعدی
        last_block = self.conv3[0]  # Conv3d
        return last_block.out_channels


In [ ]:
#@title تست
#@markdown

'''
assert image_aug.shape[1] == 4, f"expected 4 modalities, got {image_aug.shape}"

def check_levels(x0,x1,x2, base):
    assert x0.shape[1] == base
    assert x1.shape[1] == base
    assert x2.shape[1] == base
    assert x0.shape[2:] == (224,224,160)
    assert x1.shape[2:] == (112,112,80)
    assert x2.shape[2:] == (56,56,40)


t1  = image_aug[:, 0:1]
t1gd  = image_aug[:, 1:2]
t2    = image_aug[:, 2:3]
flair = image_aug[:, 3:4]

Cnvstm = ConvStem(in_channels=1, base_channels=BASE, return_all=True).to(device)
# عبور هر مدالیته
t1_x0, t1_x1, t1_x2       = Cnvstm(t1)
t1gd_x0, t1gd_x1, t1gd_x2 = Cnvstm(t1gd)
t2_x0, t2_x1, t2_x2       = Cnvstm(t2)
fl_x0, fl_x1, fl_x2       = Cnvstm(flair)

#print("stem out_channels:", Cnvstm.out_channels)
#print(len(x))

#check_levels(t1_x0, t1_x1, t1_x2, base)

print("T1 levels: '\n",    list(t1_x0.detach().shape), list(t1_x1.detach().shape), list(t1_x2.detach().shape))
print("T1Gd levels: '\n",  list(t1gd_x0.detach().shape), list(t1gd_x1.detach().shape), list(t1gd_x2.detach().shape))
print("T2 levels: '\n",    list(t2_x0.detach().shape), list(t2_x1.detach().shape), list(t2_x2.detach().shape))
print("FLAIR levels: '\n", list(fl_x0.detach().shape), list(fl_x1.detach().shape), list(fl_x2.detach().shape))
'''


'\nassert image_aug.shape[1] == 4, f"expected 4 modalities, got {image_aug.shape}"\n\ndef check_levels(x0,x1,x2, base):\n    assert x0.shape[1] == base\n    assert x1.shape[1] == base\n    assert x2.shape[1] == base\n    assert x0.shape[2:] == (224,224,160)\n    assert x1.shape[2:] == (112,112,80)\n    assert x2.shape[2:] == (56,56,40)\n\n\nt1  = image_aug[:, 0:1]\nt1gd  = image_aug[:, 1:2]\nt2    = image_aug[:, 2:3]\nflair = image_aug[:, 3:4]\n\nCnvstm = ConvStem(in_channels=1, base_channels=BASE, return_all=True).to(device)\n# عبور هر مدالیته\nt1_x0, t1_x1, t1_x2       = Cnvstm(t1)\nt1gd_x0, t1gd_x1, t1gd_x2 = Cnvstm(t1gd)\nt2_x0, t2_x1, t2_x2       = Cnvstm(t2)\nfl_x0, fl_x1, fl_x2       = Cnvstm(flair)\n\n#print("stem out_channels:", Cnvstm.out_channels)\n#print(len(x))\n\n#check_levels(t1_x0, t1_x1, t1_x2, base)\n\nprint("T1 levels: \'\n",    list(t1_x0.detach().shape), list(t1_x1.detach().shape), list(t1_x2.detach().shape))\nprint("T1Gd levels: \'\n",  list(t1gd_x0.detach().shape

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AxialAttention(nn.Module):
    """
    Axial Self-Attention روی یکی از محورها (D یا H یا W).
    بهبودها:
      - assert: dim % num_heads == 0
      - dropout روی attention و پروجکشن خروجی
      - بایاس مکانی نسبی ۱بعدی (Relative Positional Bias)
      - گزینه‌ی PreNorm (پایداری بیشتر)
      - استفاده‌ی خودکار از scaled_dot_product_attention اگر موجود باشد
    """
    def __init__(
        self,
        dim: int,
        num_heads: int = 4,
        axis: str = 'H',                 # 'D' یا 'H' یا 'W'
        attn_dropout: float = 0.0,
        # دراپ‌اوتی که روی وزن‌های توجه اعمال میشه
        # جلوگیری از overfitting توجه‌ها؛ نذاریم یک موقعیت همیشه ۱۰۰٪ وزن بگیره.
        # معمولاً 0.0 ~ 0.1 خوبه. اول 0.0 شروع کن، اگر overfit دیدی 0.05 یا 0.1.
        proj_dropout: float = 0.0,
        # دراپ‌اوتی که بعد از ادغام headها و پروجکشن خروجی اعمال میشه.
        # Regularize ترکیب headها (مثل دراپ‌اوت بعد از nn.Linear در ترنسفورمر).
        # معمولاً 0.0 ~ 0.1. اگر FFN هم دراپ‌اوت داره، جمع کل رو خیلی بالا نبر.
        # به attention «حس فاصله» در آن محور می‌دهد (نزدیک/دور) بدون موقعیت مطلق.
        #  کیفیت بهتری با هزینه‌ی ناچیز (پارامترهای کمی: heads × (2*max_len-1)).
        # یک مقدار ثابت مثل 128 یا 256 برای همه استیج‌ها.

        prenorm: bool = False,
        use_rel_bias: bool = True,
        max_len: int = 256,              # باید >= طول محور در استیج‌های شما باشد (مثلاً 56/40)، 256 امن است
        # حداکثر طول دنباله برای Relative Positional Bias یک‌بعدی.
        # موقع ساخت جدول بایاس استفاده میشه
    ):
        super().__init__()
        assert axis in ['D', 'H', 'W'], "axis باید یکی از {'D','H','W'} باشد"
        assert dim % num_heads == 0, "dim باید بر num_heads بخش‌پذیر باشد"
        # تا کانال‌ها بین هدها تقسیم شوند

        self.axis = axis
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        # در attention، ضرب داخلی Q و K می‌تواند بزرگ شود؛ تقسیم بر √d (اینجا head_dim) جلوی softmax بسیار تیز (و گرادیان ناپایدار) را می‌گیرد.
        self.prenorm = prenorm
        # اگر True باشد، قبل از attention روی ورودی LayerNorm می‌زنیم (پایداری بیشتر در بعضی تنظیمات).
        self.use_rel_bias = use_rel_bias
        self.max_len = max_len

        self.qkv  = nn.Linear(dim, dim * 3, bias=True)
        # یک لایه‌ی خطی که به‌صورت مشترک Q/K/V را می‌سازد (کارایی بهتر از سه Linear جدا در PyTorch).
        self.proj = nn.Linear(dim, dim, bias=True)
        # بعد از جمع هدها، کانال‌ها دوباره به فضای dim نگاشت می‌شوند (مثل 1×1 conv در CNN).
        self.drop_attn = nn.Dropout(attn_dropout)
        self.drop_proj = nn.Dropout(proj_dropout)
        self.norm = nn.LayerNorm(dim)
        # نرمال‌سازی روی محور کانال؛ بسته به prenorm، قبل یا بعد از Residual اعمال می‌شود.


        # تعریف پارامترِ بایاس نسبی
        '''
        self.rel_bias: یک جدولِ بایاس یادگرفتنی به اندازهٔ (h, 2*max_len - 1)
        ؛ برای هر head  یک بردار طول 2 * max_len-1.
        چرا 2*max_len-1؟ چون اختلاف موقعیت‌ها روی یک محور j - i می‌تواند از -(L-1) تا +(L-1) باشد (L طول دنباله). این بازه طولش 2L-1 است. ما با max_len یک سقف امن می‌گذاریم تا برای همهٔ Lهای ممکن در آن استیج کافی باشد.
        zeros_ یعنی در شروعِ آموزش، بدون بایاس آغاز می‌کنیم (اثر صفر). مدل اگر مفید باشد، طی آموزش خودش بایاس‌های مناسب را یاد می‌گیرد؛ شروع امن و پایدار.
        '''
        if use_rel_bias:
            self.rel_bias = nn.Parameter(torch.zeros(num_heads, 2 * max_len - 1))
            nn.init.zeros_(self.rel_bias)
        '''
        برای هر head یک بردار بایاس ۱بعدی داریم که به اختلاف موقعیت‌ها (j−i) وابسته است.
        طولش 2L-1 است (از −(L−1) تا +(L−1)).
        این بایاس به ماتریس امتیاز attention جمع می‌شود تا مدل فرق «نزدیک/دور» را در همان محور بفهمد (خیلی مؤثر با هزینه‌ی ناچیز).

        self.rel_bias: یک پارامتر یادگرفتنی برای هر head با طول 2*max_len-1 (چون اختلاف نسبی j−i در بازه‌ی [-(L-1), +(L-1)] است که طولش 2L-1 می‌شود؛ ما سقف امن max_len می‌گذاریم).
        با صفر شروع می‌کنیم تا در ابتدای آموزش، اتنشن «بی‌بایاس» باشد؛ مدل خودش یاد می‌گیرد چه فاصله‌هایی مهم‌ترند.
        '''

        # تشخیص خودکار وجود SDPA (PyTorch>=2)
        self._has_sdpa = hasattr(F, "scaled_dot_product_attention")
        # پیاده‌سازی fused و بهینه‌ی attention که معمولاً سریع‌تر/کم‌حافظه‌تر از matmul دستی است. اگر موجود باشد، از آن استفاده می‌کنیم.

    def _seq_reshape(self, x):
        """
        x: (B,C,D,H,W)  →  x_seq: (B*, L, C)  و اطلاعات موردنیاز برای بازگردانی شکل
        """
        B, C, D, H, W = x.shape
        # Axial یعنی فقط روی یک محور (مثلاً H) توالی بسازیم. پس باید تنسور را طوری permute کنیم که «طولِ دنباله» همان محور باشد (L=H) و بقیه‌ی ابعاد به batch مؤثر ضرب شوند (B*D*W).

        if self.axis == 'H':
            x_seq = x.permute(0, 2, 4, 3, 1).contiguous().view(B * D * W, H, C)  # (B*, L=H, C)
            '''
            permute فقط view را تغییر می‌دهد؛ اما پراکندگی حافظه را عوض می‌کند.
            contiguous() اطمینان می‌دهد داده‌ها به‌ترتیب جدید در حافظه پیوسته‌اند تا view قانونی باشد.
            '''
            L = H
            '''
            پس L طول توالی روی همان محور انتخابی است. مثال شما: ورودی (1,32,56,56,40)
            ورودی ۵بعدی (B,C,D,H,W)
            محور H: L=56
            محور W: L=40
            محور D: L=56
            '''
            shape_back = (B, D, W, H, C)
            back_perm = (0, 4, 1, 3, 2)
            # اطلاعات لازم برای برگرداندن خروجی از فضای توالی به شکل اصلی (B,C,D,H,W).

        elif self.axis == 'W':
            x_seq = x.permute(0, 2, 3, 4, 1).contiguous().view(B * D * H, W, C)  # (B*, L=W, C)
            L = W
            shape_back = (B, D, H, W, C)
            back_perm = (0, 4, 1, 2, 3)
        else:  # 'D'
            x_seq = x.permute(0, 3, 4, 2, 1).contiguous().view(B * H * W, D, C)  # (B*, L=D, C)
            L = D
            shape_back = (B, H, W, D, C)
            back_perm = (0, 4, 3, 1, 2)
        return x_seq, L, shape_back, back_perm

    '''
    برای هر اختلاف فاصله‌ی j−i، یک بایاس یادگرفتنی داریم. این بایاس به امتیاز خام attention جمع می‌شود قبل از softmax.
    چرا مفید است؟ Attention خام permutation-invariant است؛ بایاس نسبی به آن «حس فاصله» می‌دهد (بدون موقعیت مطلق)، که برای تصاویر/حجم‌ها مؤثر است.
    '''
    def _add_rel_bias(self, attn, L, device):
        """
        attn: (B*, h, L, L)   →   attn + relative_bias
        """
        # ایمنی: اطمینان از اینکه L از max_len بیشتر نیست
        # برای هر head (h)، ماتریس L×L از امتیازها.
        assert L <= self.max_len, f"طول محور {L} > max_len={self.max_len}. مقدار max_len را بزرگ‌تر بگیر."
        idx = torch.arange(L, device=device)   # [0,1,2,...,L-1]

        # ماتریس اختلافها: j-i ∈ [-(L-1)..(L-1)]  → shift به [0..2L-2]
        # با Broadcast ساده، ماتریس اختلاف نسبی می‌سازیم: در خانه‌ی (i,j)، مقدار j - i.
        # چون منفی هم می‌شود، با +(L-1) آن را به بازه‌ی ایندکس غیرمنفی [0 .. 2L-2] می‌بریم.
        rel = (idx[None, :] - idx[:, None]) + (L - 1)  # (L,L)

        # انتخاب بایاس برای هر head
        # از جدول ۱بعدیِ بایاس برای هر head، بر اساس ماتریس ایندکس‌های rel، یک ماتریس بایاس (h,L,L) در می‌آوریم؛ یعنی برای هر اختلاف نسبی، وزن مخصوصش.
        bias = self.rel_bias[:, rel]                   # (h, L, L)

        return attn + bias.unsqueeze(0)                # (B*, h, L, L) + (1, h, L, L)

        # attn همان امتیاز خام (QKᵀ * scale) است (قبل از softmax).
        # bias.unsqueeze(0) یک بُعد batch مؤثر اضافه می‌کند تا روی B* broadcast شود (برای تمام توالی‌ها یکی است)
        # جمع کردن این بایاس با لوگیت‌ها یعنی قبل از softmax احتمال‌ها را به نفع فاصله‌های خاص کمی بالا/پایین می‌بریم.
        '''
        _add_rel_bias به اتنشن یاد می‌دهد چقدر به همسایه‌های نزدیک/دور نگاه کند—برای هر head و فقط بر حسب فاصلهٔ نسبی—و با همین «اشارهٔ کوچک» کیفیت توجه و در نتیجه نمایش ویژگی‌ها بهتر می‌شود.
        قدم به قدم:
                طول دنباله (L) را می‌گیریم (مثلاً H=56).
                یک ماتریس می‌سازیم که هر خانه‌اش «فاصلهٔ نسبی» بین دو موقعیت است:
                سطر i، ستون j → Δ = j − i
                Δ در بازهٔ [−(L−1) .. +(L−1)] است.
                چون Δ منفی هم می‌شود، با یک شیفت ساده آن را به اندیس غیرمنفی نگاشت می‌کنیم تا بتوانیم از جدول بایاس بخوانیم.
                برای هر head یک بردار بایاس داریم (اندازه‌اش 2L−1). با اندیس‌های مرحلهٔ قبل، یک ماتریس بایاس (L×L) می‌سازیم.
                این ماتریس بایاس را به امتیازهای اتنشن (قبل از softmax) جمع می‌کنیم. بعد softmax می‌زنیم.
                نتیجه:
                اگر مدل یاد بگیرد «همسایه‌های خیلی نزدیک مهم‌ترند»، بایاسِ Δ=±1 را مثبت می‌کند → وزن توجه به همسایه‌ها بیشتر می‌شود.
                اگر یاد بگیرد «یه فاصلهٔ خاص (مثلاً Δ≈3) اغلب مفید است»، همان اندیس بایاس را بالا می‌برد.
                اگر فاصله‌ای معمولاً بی‌ربط باشد، بایاس منفی می‌شود و وزنش کم می‌شود.
                احتمال‌ها «جهت‌دار» می‌شوند
        '''

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, C=dim, D, H, W) → خروجی با همان شکل
        """
        B, C, D, H, W = x.shape
        x_seq, L, shape_back, back_perm = self._seq_reshape(x)       # (B*, L, C)
        # شکل (B*, L, C)؛ حالا «توالی محوری» داریم.
        residual = x_seq
        # برای اتصال رزیدوال (skip) نگه‌اش می‌داریم.

        if self.prenorm:
            x_seq = self.norm(x_seq)
        # اول نرمال می‌کنیم، بعد Q/K/V می‌سازیم؛ معمولاً گرادیان پایدارتر می‌شود (در مدل‌های عمیق‌تر).

        # Q, K, V
        qkv = self.qkv(x_seq)                                        # (B*, L, 3C)
        q, k, v = qkv.chunk(3, dim=-1)
        Bq = q.shape[0]
        q = q.view(Bq, L, self.num_heads, self.head_dim).transpose(1, 2)  # (B*, h, L, d)
        k = k.view(Bq, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(Bq, L, self.num_heads, self.head_dim).transpose(1, 2)

        if self._has_sdpa:
            # SDPA مقیاس‌دهی را خودش انجام می‌دهد؛ بایاس نسبی را به عنوان attn_mask اضافه می‌کنیم
            attn_mask = None
            # اینجا ما «ماسک» را به‌عنوان بایاس افزوده‌شونده به امتیازها استفاده می‌کنیم (همون relative bias per-head).
            if self.use_rel_bias:
                assert L <= self.max_len, f"L={L} exceeds max_len={self.max_len}; increase max_len."
                # (1, h, L, L) → broadcast روی B*
                idx = torch.arange(L, device=x.device)
                rel = (idx[None, :] - idx[:, None]) + (L - 1)
                bias = self.rel_bias[:, rel]  # (h,L,L)
                attn_mask = bias.unsqueeze(0) # (1,h,L,L)
                # تا شکل (1,h,L,L) شده و روی Batch مؤثر broadcast شود.
                '''
                با unsqueeze(0) یک بُعد batch به ابتدای آن اضافه می‌کنیم تا شکلش با SDPA سازگار شود:
                SDPA می‌تونه attn_mask‌ی با شکلی broadcastable به (B*, h, L, L) بپذیره.
                اینجا (1,h,L,L) روی بُعد B* به‌صورت خودکار broad­cast می‌شود (برای همهٔ دنباله‌ها یکی است).
                '''
            out = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=attn_mask,             # بایاس افزایشی
                dropout_p=self.drop_attn.p if self.training else 0.0,
                is_causal=False
            )                                     # (B*, h, L, d)
            '''
            این تابع پشت‌صحنه کارهای زیر را انجام می‌دهد:
                    محاسبهٔ امتیازها: scores = q @ k^T / sqrt(d)
                    اگر attn_mask عددی باشد → جمع افزایشی: scores += attn_mask
                    یعنی مثلاً اگر بایاس برای فاصلهٔ کم مثبت یاد گرفته شود، امتیاز آن جفت (i,j) قبل از softmax بالاتر می‌رود ⇒ احتمال توجه بیشتر.
                    softmax روی محور آخر (L)
                    dropout روی ماتریس attention (فقط در train)
                    ضرب در v: attn @ v
                    خروجی شکلش می‌شود (B*, h, L, d).
            '''
            out = out.transpose(1, 2).reshape(Bq, L, self.dim)       # (B*, L, C)
            # ادغام هدها: از (B*, h, L, d) به (B*, L, h*d=dim).
        else:
            # مسیر کلاسیک
            attn = (q @ k.transpose(-2, -1)) * self.scale            # (B*, h, L, L)
            if self.use_rel_bias:
                attn = self._add_rel_bias(attn, L, device=x.device)
            attn = F.softmax(attn, dim=-1)
            attn = self.drop_attn(attn)
            out = (attn @ v).transpose(1, 2).reshape(Bq, L, self.dim)

        # پروجکشن و رزیدوال + نرمال‌سازی
        out = self.proj(out)
        '''
        دقیقاً معادل Conv3d با kernel=1×1×1 است (وقتی برش را دوباره به 3D برگردانی).
        نقش‌اش: میکس دوبارهٔ اطلاعات headها و ساخت یک فضای ویژگی «پس از توجه». هر head بخشی از الگوها را می‌بیند؛ این لایه یاد می‌گیرد چطور سهم هر head را ترکیب کند.
        چرا لازم است؟ در Multi-Head Attention، بعد از concat باید یک نگاشت یادگرفتنی برای ترکیب headها داشته باشیم؛ وگرنه صرفاً چسباندن کانال‌ها بدون یادگیری وزن‌های ترکیب است.
        '''
        out = self.drop_proj(out)
        # در train: به‌طور تصادفی بخشی از عناصر خروجی را صفر می‌کند (با احتمال proj_dropout)، سپس مقیاس‌دهی مناسب انجام می‌شود تا امید ریاضی ثابت بماند.
        out = out + residual
        if not self.prenorm:
            out = self.norm(out)
        # بعد از ترکیب headها با proj (مثل 1×1 conv)، residual را اضافه می‌کنیم و اگر PreNorm نبود، Post-Norm می‌زنیم.

        # بازگرداندن به (B,C,D,H,W)
        out = out.view(*shape_back).permute(*back_perm).contiguous()
        # با همان اطلاعات shape_back و back_perm، خروجی را به فضای 3D برمی‌گردانیم؛ contiguous() برای اطمینان از چیدمان حافظه.
        return out


In [ ]:
#@title عنوانِ این سلول
#@markdown (اختیاری) توضیح کوتاه اینجاست

'''
چرا از SDPA استفاده می‌کنیم؟
معمولاً سریع‌تر و کم‌مصرف‌تر از پیاده‌سازی دستی (matmul + softmax + …) است؛
'''

'''
اگر حس کردی مدل خیلی وابسته به یک/دو head شده (attention maps تک‌خطی، overfit)، کمی proj_dropout را بالا ببر.

اگر آموزش ناپایدار شد یا loss نوسان می‌کند، proj_dropout را پایین نگه‌دار و اول prenorm=True یا attn_dropout را تیون کن.

برای سرعت/حافظه، این بخش هزینه‌ی ناچیزی دارد؛ bottle-neck اصلی همان ماتریس‌های attention است، نه این Linear.
'''

'''
پیشنهادهای کوچک (اختیاری)

اگر جایی طول محور از 256 بیشتر شد، فقط max_len را بزرگ کن (مثلاً 384).
اگر آموزش کمی ناپایدار بود: prenorm=True و/یا attn_dropout=proj_dropout=0.1.
برای سرعت بیشتر (اگر لازم شد): num_heads=2 یا فقط دو محور (مثلاً D و H) در برخی استیج‌ها
مقدار max_len را طوری بگذار که از طول محور در استیج‌های شما بزرگ‌تر باشد (برای 56/40 روی H/W/D، مقدار 256 کفایت دارد).
اگر جایی آموزش ناپایدار شد، prenorm=True را امتحان کن و/یا attn_dropout / proj_dropout را کمی بالا ببر (مثلاً 0.1).
اگر PyTorch≤1.x داری و SDPA موجود نیست، مسیر کلاسیک به‌صورت خودکار استفاده می‌شود.
'''
#  تیونینگ هدها و دراپ‌اوت: اگر کند شد یا OOM داشتی، num_heads را کم کن (مثلاً 2) و attn_dropout/proj_dropout را 0.1 بگذار.

'''
x: [B, C, D, H, W] = [1, 32, 56, 56, 40]
num_heads = 4 → هر head ‌کانال: head_dim = 32 / 4 = 8
**scale = 1/√head_dim = 1/√8 ≈ 0.3536`
max_len = 256
prenorm=False، attn_dropout=proj_dropout=0.0 و SDPA نداریم
معمولاً ترتیب D → H → W را انجام می‌دهد. هر پاس دقیقاً همین منطق را دارد ولی L (طول توالی) و تعداد توالی‌ها (B*…) عوض می‌شود
1) تبدیل 5بعدی به «دنباله‌های 1بعدی»
            x_seq = x.permute(0, 3, 4, 2, 1).contiguous().view(B*H*W, D, C)
            ورودی: (1, 32, 56, 56, 40)
            permute → (B, H, W, D, C) = (1, 56, 40, 56, 32)
            view → x_seq: (B*H*W, L=D, C) = (1*56*40, 56, 32) = (2240, 56, 32)
2) تولید Q/K/V و جدا کردن headها
            qkv = Linear(32 → 96)(x_seq)  # (2240, 56, 96)
            q,k,v = chunk(..., 3)         # هرکدام (2240, 56, 32)
            q = reshape → (2240, 4, 56, 8)
            k = reshape → (2240, 4, 56, 8)
            v = reshape → (2240, 4, 56, 8)
3) امتیاز توجه و softmax
            attn = (q @ k.transpose(-2, -1)) * scale
            # attn: (2240, 4, 56, 56)
            برای هر head و هر توالی، یک ماتریس 56×56 می‌سازیم (شباهت همه‌ی توکن‌ها با هم).
            اگر relative positional bias فعال باشد، یک بایاس شکل (h=4, L=56, L=56) به این امتیاز اضافه می‌شود (بر اساس اختلاف j−i).
            softmax روی محور آخر:
                attn = softmax(attn, dim=-1)  # همان (2240, 4, 56, 56)
4) ترکیب با V و ادغام headه
            out = (attn @ v)                       # (2240, 4, 56, 8)
            out = transpose/reshape → (2240, 56, 32)
            out = proj: Linear(32→32) → (2240, 56, 32)
            out = out + residual (2240, 56, 32)
            out = LayerNorm → (2240, 56, 32)   # چون prenorm=False

            Residual: ورودی دنباله‌ای + خروجی attention → پایدارتر.
            LayerNorm: نرمال‌سازی کانالی بعد از residual.
5) برگشت به شکل 5بعدی
            out = out.view(B, H, W, D, C).permute(0, 4, 3, 1, 2)
            # → (1, 32, 56, 56, 40)  ← دقیقاً همان شکل ورودی
'''
'''
جمع‌بندی شکل‌ها و هزینه:
            تعداد دنباله‌ها در هر محور = ضرب دو محور دیگر × B:
            محور D: B*H*W = 1*56*40 = 2240 توالیِ طول 56
            محور H: B*D*W = 1*56*40 = 2240 توالیِ طول 56
            محور W: B*D*H = 1*56*56 = 3136 توالیِ طول 40
نقش Relative Positional Bias
            برای محور D/H: L=56 → بایاس با شکل (heads=4, 2L−1=111) که به ماتریس (4,56,56) نگاشت می‌شود.
            برای محور W: L=40 → بایاس (4,79) به (4,40,40) نگاشت می‌شود.
            این بایاس (وابسته به j−i) کمک می‌کند مدل بفهمد «نزدیک‌تر/دورتر» بودن در همان محور چه اثری روی توجه دارد — بدون نیاز به مختصات مطلق
'''

'''
نکتهٔ عملی
برای ورودی شما (stage با ابعاد 56×56×40)، مقدار امن:     max_len = 128  # یا 256
'''

'''
شکل‌ها یک‌بار با مثال شما
ورودی: (B,C,D,H,W) = (1,32,56,56,40)، محور H:
بعد از reshape: q,k,v: (B*, h, L, d) = (1×56×40=2240, 4, 56, 8)
attn_mask: (1, 4, 56, 56) → broadcast به (2240, 4, 56, 56)
خروجی SDPA: (2240, 4, 56, 8) → ادغام: (2240, 56, 32) → برگشت به (1, 32, 56, 56, 40)
'''

'\nشکل\u200cها یک\u200cبار با مثال شما\nورودی: (B,C,D,H,W) = (1,32,56,56,40)، محور H:\nبعد از reshape: q,k,v: (B*, h, L, d) = (1×56×40=2240, 4, 56, 8)\nattn_mask: (1, 4, 56, 56) → broadcast به (2240, 4, 56, 56)\nخروجی SDPA: (2240, 4, 56, 8) → ادغام: (2240, 56, 32) → برگشت به (1, 32, 56, 56, 40)\n'

In [ ]:
import torch
import torch.nn as nn

'''
کِی از FFN صرف‌نظر کنم؟
اگر GPU ضعیف است یا عمق شبکه زیاد است، موقتاً use_ffn=False بگذار؛ بعد از پایدار شدن آموزش، روشن کن.
اگر روشن است و overfit داری، کمی ffn_dropout بده (مثلاً 0.1).
'''
class FeedForward1x1(nn.Module):
    """
    FFN سبکِ مکانی (per-voxel): Conv1x1 → GELU → Dropout → Conv1x1 + Residual + LayerNorm
    """
    def __init__(self, dim: int, expansion: int = 2, dropout: float = 0.0, prenorm: bool = False):
        super().__init__()
        hidden = dim * expansion
        self.prenorm = prenorm
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Conv3d(dim, hidden, kernel_size=1, bias=True)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Conv3d(hidden, dim, kernel_size=1, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,C,D,H,W)  — LayerNorm روی محور C انجام می‌شود: قبلش به (B,D,H,W,C) می‌بریم و برمی‌گردونیم
        res = x
        y = x
        if self.prenorm:
            y = y.permute(0,2,3,4,1).contiguous()
            y = self.norm(y)
            y = y.permute(0,4,1,2,3).contiguous()

        y = self.fc1(y)
        y = self.act(y)
        y = self.drop(y)
        y = self.fc2(y)
        y = self.drop(y)
        y = y + res

        if not self.prenorm:
            y_n = y.permute(0,2,3,4,1).contiguous()
            y_n = self.norm(y_n)
            y = y_n.permute(0,4,1,2,3).contiguous()
        return y


class SelfModalAxialBlock(nn.Module):
    """
    بلاک Self-Axial برای یک مدالیته:
      AxialAttention(D) → AxialAttention(H) → AxialAttention(W) → (اختیاری) FFN1x1
    نکات:
      - هر AxialAttention خودش residual+LayerNorm دارد (prenorm قابل تنظیم).
      - FFN سبک اختیاری برای پالایش ویژگی‌ها.
    """
    def __init__(
        self,
        dim: int,
        num_heads: int = 4,
        attn_dropout: float = 0.0,
        proj_dropout: float = 0.0,
        prenorm_attn: bool = False,      # اگر آموزش ناپایدار بود، True کن
        use_rel_bias: bool = True,
        max_len: int = 256,              # >= بیشترین طول محور در این استیج
        use_ffn: bool = True,
        ffn_expansion: int = 2,
        ffn_dropout: float = 0.0,
        ffn_prenorm: bool = False
    ):
        super().__init__()
        # سه محور
        self.attn_d = AxialAttention(dim, num_heads=num_heads, axis='D',
                                     attn_dropout=attn_dropout, proj_dropout=proj_dropout,
                                     prenorm=prenorm_attn, use_rel_bias=use_rel_bias, max_len=max_len)

        self.attn_h = AxialAttention(dim, num_heads=num_heads, axis='H',
                                     attn_dropout=attn_dropout, proj_dropout=proj_dropout,
                                     prenorm=prenorm_attn, use_rel_bias=use_rel_bias, max_len=max_len)

        self.attn_w = AxialAttention(dim, num_heads=num_heads, axis='W',
                                     attn_dropout=attn_dropout, proj_dropout=proj_dropout,
                                     prenorm=prenorm_attn, use_rel_bias=use_rel_bias, max_len=max_len)

        self.use_ffn = use_ffn
        self.ffn = FeedForward1x1(dim, expansion=ffn_expansion,
                                  dropout=ffn_dropout, prenorm=ffn_prenorm) if use_ffn else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C=dim, D, H, W)
        x = self.attn_d(x)
        x = self.attn_h(x)
        x = self.attn_w(x)
        x = self.ffn(x)
        return x



'''
برای استیج 1/4 (H=56, W=56, D=40) مقدار max_len=256 امن است؛ اگر جایی بزرگ‌تر شد، عدد را بالاتر بگذار.
اگر دیدی آموزش ناپایدار است: prenorm_attn=True یا ffn_prenorm=True و کمی attn_dropout/proj_dropout را بالا ببر (مثلاً 0.1).
اگر سرعت برات مهم شد: تعداد headها را کم کن (num_heads=2) یا فقط دو محور (مثلاً D و H) را استفاده کن.
'''

'\nبرای استیج 1/4 (H=56, W=56, D=40) مقدار max_len=256 امن است؛ اگر جایی بزرگ\u200cتر شد، عدد را بالاتر بگذار.\nاگر دیدی آموزش ناپایدار است: prenorm_attn=True یا ffn_prenorm=True و کمی attn_dropout/proj_dropout را بالا ببر (مثلاً 0.1).\nاگر سرعت برات مهم شد: تعداد headها را کم کن (num_heads=2) یا فقط دو محور (مثلاً D و H) را استفاده کن.\n'

In [ ]:
#@title تست
#@markdown

'''
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# ورودی تک‌مدالیته با سایز اصلی (224,224,160)
x = torch.randn(1, 1, 224, 224, 160, device=device)

# 1) خروجی ConvStem → باید (1, 32, 56, 56, 40) باشد
stem = ConvStem(in_channels=1, base_channels=32).to(device)
x2 = stem(x)
if isinstance(x2, (tuple, list)):   # اگر نسخه شما tuple برگرداند
    x2 = x2[-1]
print("ConvStem out:", tuple(x2.shape))  # انتظار: (1, 32, 56, 56, 40)



# 2) AxialAttention برای هر محور (شکل باید ثابت بماند)
attn_D = AxialAttention(dim=32, num_heads=4, axis='D', max_len=256).to(device)
attn_H = AxialAttention(dim=32, num_heads=4, axis='H', max_len=256).to(device)
attn_W = AxialAttention(dim=32, num_heads=4, axis='W', max_len=256).to(device)

with torch.no_grad():
    yD = attn_D(x2)
    yH = attn_H(x2)
    yW = attn_W(x2)

print("Axial D:", tuple(yD.shape))  # (1, 32, 56, 56, 40)
print("Axial H:", tuple(yH.shape))  # (1, 32, 56, 56, 40)
print("Axial W:", tuple(yW.shape))  # (1, 32, 56, 56, 40)



# 2) تست SelfModalAxialBlock (D→H→W + اختیاری FFN داخلش)
sblk = SelfModalAxialBlock(dim=32, num_heads=4, max_len=256).to(device)
with torch.no_grad():
    y_sblk = sblk(x2)
print("SelfModalAxialBlock:", tuple(x2.shape), "->", tuple(y_sblk.shape))  # همان شکل

# 3) تست FeedForward1x1 به‌تنهایی
ffn = FeedForward1x1(dim=32, expansion=2, dropout=0.0, prenorm=False).to(device)
with torch.no_grad():
    y_ffn = ffn(x2)
print("FeedForward1x1:", tuple(x2.shape), "->", tuple(y_ffn.shape))  # همان شکل
'''

'\nimport torch\n\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\n\n# ورودی تک\u200cمدالیته با سایز اصلی (224,224,160)\nx = torch.randn(1, 1, 224, 224, 160, device=device)\n\n# 1) خروجی ConvStem → باید (1, 32, 56, 56, 40) باشد\nstem = ConvStem(in_channels=1, base_channels=32).to(device)\nx2 = stem(x)\nif isinstance(x2, (tuple, list)):   # اگر نسخه شما tuple برگرداند\n    x2 = x2[-1]\nprint("ConvStem out:", tuple(x2.shape))  # انتظار: (1, 32, 56, 56, 40)\n\n\n\n# 2) AxialAttention برای هر محور (شکل باید ثابت بماند)\nattn_D = AxialAttention(dim=32, num_heads=4, axis=\'D\', max_len=256).to(device)\nattn_H = AxialAttention(dim=32, num_heads=4, axis=\'H\', max_len=256).to(device)\nattn_W = AxialAttention(dim=32, num_heads=4, axis=\'W\', max_len=256).to(device)\n\nwith torch.no_grad():\n    yD = attn_D(x2)\n    yH = attn_H(x2)\n    yW = attn_W(x2)\n\nprint("Axial D:", tuple(yD.shape))  # (1, 32, 56, 56, 40)\nprint("Axial H:", tuple(yH.shape))  # (1, 32, 56, 56, 40)\nprint("Axial W:

In [ ]:
class CrossAxialAttention(nn.Module):
    def __init__(self, dim, num_heads=4, axis='H',
                 attn_dropout=0.0, proj_dropout=0.0,
                 prenorm=False, use_rel_bias=True, max_len=256):
        super().__init__()
        assert axis in ['D','H','W']
        assert dim % num_heads == 0
        self.axis, self.dim, self.num_heads = axis, dim, num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.prenorm, self.use_rel_bias, self.max_len = prenorm, use_rel_bias, max_len

        # Q از x_q، و K/V از x_kv
        self.q_proj = nn.Linear(dim, dim, bias=True)
        # نگاشت ویژگی‌های A (x_q) به فضای Q.
        self.kv_proj = nn.Linear(dim, dim * 2, bias=True)
        # نگاشت ویژگی‌های B (x_kv) به K و V همزمان (خروجی 2×dim و بعد chunk می‌کنیم).
        # Q از x_q (مثلاً T1) و K,V از x_kv (مثلاً T1Gd).
        # این یعنی ویژگی‌های A یاد می‌گیرند «به کجای B» نگاه کنند.
        # این‌ها دقیقاً مثل Conv3d(1×1×1) روی هر توکن (هر موقعیت از توالی) عمل می‌کنند: یک تبدیل کانالیِ قابل‌آموزش.

        self.proj = nn.Linear(dim, dim, bias=True)
        # بعد از ادغام headها، ترکیب نهایی (مثل Conv1×1×1).
        self.drop_attn = nn.Dropout(attn_dropout)
        self.drop_proj = nn.Dropout(proj_dropout)
        self.norm_q = nn.LayerNorm(dim)  # فقط روی توالی Q

        if use_rel_bias:
            self.rel_bias = nn.Parameter(torch.zeros(num_heads, 2*max_len - 1))
            nn.init.zeros_(self.rel_bias)

        self._has_sdpa = hasattr(F, "scaled_dot_product_attention")

    def _seq_reshape(self, x):
        B,C,D,H,W = x.shape
        if self.axis=='H':
            x_seq = x.permute(0,2,4,3,1).contiguous().view(B*D*W, H, C)
            L=H
            back=(B,D,W,H,C)
            perm=(0,4,1,3,2)
        elif self.axis=='W':
            x_seq = x.permute(0,2,3,4,1).contiguous().view(B*D*H, W, C)
            L=W
            back=(B,D,H,W,C)
            perm=(0,4,1,2,3)
        else:
            x_seq = x.permute(0,3,4,2,1).contiguous().view(B*H*W, D, C)
            L=D
            back=(B,H,W,D,C)
            perm=(0,4,3,1,2)
        return x_seq, L, back, perm

    def _rel_bias(self, L, device):
        assert L <= self.max_len, f"L={L} exceeds max_len={self.max_len}; increase max_len."
        idx = torch.arange(L, device=device)
        rel = (idx[None,:] - idx[:,None]) + (L-1)  # (L,L)→[0..2L-2]
        return self.rel_bias[:, rel]  # (h,L,L)
        # چون توجه «کراس» هم باید بداند فاصله‌های نسبی روی همان محور چه گرایشی دارند (همسایهٔ نزدیک‌تر اغلب مفیدتر است).

    def forward(self, x_q, x_kv):
        # reshape دنباله‌ها روی محور انتخابی
        q_seq, L, back, perm = self._seq_reshape(x_q)     # (B*, L, C)
        kv_seq, _, _, _      = self._seq_reshape(x_kv)    # (B*, L, C)
        res = q_seq
        if self.prenorm:
            q_seq = self.norm_q(q_seq)

        # Q از q_seq ، K/V از kv_seq
        Q = self.q_proj(q_seq)                             # (B*, L, C)
        KV = self.kv_proj(kv_seq)                          # (B*, L, 2C)
        K, V = KV.chunk(2, dim=-1)

        Bq = Q.shape[0]                                    # = B*
        h, d = self.num_heads, self.head_dim               # d = dim / h
        # هر head بُعد d دارد و الگوی متفاوتی یاد می‌گیرد (یکی محلی‌تر، یکی دوربردتر…).

        Q = Q.view(Bq, L, h, d).transpose(1,2)             # (B*, h, L, d)
        # view(B*, L, h, d) کانال‌ها را بین headها پخش می‌کند.
        # transpose(1,2) به شکل استاندارد SDPA/Attention می‌رسیم: *(B, h, L, d)**؛
        # بعد 0: بچ مؤثر
        # بعد 1 : head
        # بعد 2: طول توالی
        # بعد 3: بُعد هر head
        K = K.view(Bq, L, h, d).transpose(1,2)
        V = V.view(Bq, L, h, d).transpose(1,2)

        if hasattr(F, "scaled_dot_product_attention"):
            attn_mask = None
            if self.use_rel_bias:
                bias = self._rel_bias(L, device=Q.device).unsqueeze(0)  # (1,h,L,L)
                attn_mask = bias
            out = F.scaled_dot_product_attention(
                Q, K, V,
                attn_mask=attn_mask,
                dropout_p=self.drop_attn.p if self.training else 0.0,
                is_causal=False
            )                                              # (B*, h, L, d)
            out = out.transpose(1,2).reshape(Bq, L, h*d)   # (B*, L, C)
        else:
            attn = (Q @ K.transpose(-2, -1)) * self.scale  # (B*, h, L, L)
            if self.use_rel_bias:
                bias = self._rel_bias(L, device=Q.device).unsqueeze(0)
                attn = attn + bias
            attn = F.softmax(attn, dim=-1)
            attn = self.drop_attn(attn)
            out = (attn @ V).transpose(1,2).reshape(Bq, L, h*d)

        out = self.proj(out)                # ترکیب یادگرفتنی headها (مثل conv1×1×1)
        out = self.drop_proj(out)
        out = out + res
        out = self.norm_q(out) if not self.prenorm else out
        out = out.view(*back).permute(*perm).contiguous()         # برگشت به (B,C,D,H,W)
        return out


In [ ]:
#@title عنوانِ این سلول
#@markdown (اختیاری) توضیح کوتاه اینجاست

'''
اتنشن خام فقط «شباهت Q و K» را می‌بیند؛ فاصله‌ی نسبی بین موقعیت‌ها را نمی‌فهمد.
ما برای هر هد یک جدول بایاس قابل‌آموزش داریم که به اختلاف نسبی  Δ=𝑗−𝑖 وزن می‌دهد.
این بایاس قبل از softmax به لگیت‌ها اضافه می‌شود تا مدل «حس فاصله» پیدا کند (مثلاً همسایه‌های نزدیک مهم‌تر شوند).
L طول دنباله روی محور انتخابی است (مثلاً اگر محور H باشد، L=H).
idx = [0, 1, ..., L-1].
rel = (j - i) + (L-1) یک ماتریس L×L از اندیس‌ها می‌سازد
بدون شیفت: j−i∈[−(L−1),...,(L−1)]
با شیفت +(L−1): می‌شود بازه‌ی غیرمنفی [0 .. 2L-2] تا بتوانیم از جدول بایاس ایندکس کنیم.
self.rel_bias شکلش (h, 2*max_len-1) است (برای هر هد، یک بردار). با ایندکس rel آن را به (h, L, L) تبدیل می‌کنیم: یک ماتریس بایاس برای هر هد.
        مثال خیلی کوچک (L = 4، یک head)
        اندیس‌ها: 0..3
        ماتریس j−i:
        با شیفت +3 → بازه‌ی 0..6.
        فرض کن مدل یاد گرفته:
        برای Δ=0 بایاس +0.6 (خود-پیکسل مهم)
        برای |Δ|=1 بایاس +0.3 (همسایه‌ی نزدیک مهم)
        برای |Δ|≥2 بایاس −0.5 (دور مهم نیست)
        نتیجه: ماتریس بایاس هر سطر/ستون را بر اساس فاصله‌ی نسبی تقویت/تضعیف می‌کند.
شکل‌ها قبل از SDPA
        پس از تبدیل محوری، برای هر محور Q/K/V شکلشان می‌شود: (B*, h, L, d)
        B* = بچِ مؤثر (مثلاً برای محور H: B×D×W)
        h = تعداد هدها
        L = طول دنباله روی آن محور
        d = dim / h
نقش attn_mask:
        bias از _rel_bias شکل (h, L, L) دارد؛ با unsqueeze(0) می‌شود (1, h, L, L) تا روی بعد B*            broadcast شود.
        در SDPA، attn_mask اگر float باشد به‌صورت افزایشی به لگیت‌ها (امتیازهای پیش از softmax) اضافه می‌شود:
                    bias+attention = scores
        بعد softmax روی محور آخر (طول L) زده می‌شود.
'''
'''
پیگیری شکل‌ها با مثال واقعی
        فرض ورودی‌های استیج: x_q.shape = x_kv.shape = (1, 32, 56, 56, 40)
        محور H:
        q_seq, kv_seq: (B*D*W, H, C) = (2240, 56, 32)، L=56
        Q,K,V: (2240, 4, 56, 8)
        logits: (2240, 4, 56, 56) (+bias) → softmax
        خروجی headها: (2240, 4, 56, 8) → ادغام → (2240, 56, 32)
        برگشت به 5بعدی: (1, 32, 56, 56, 40)
        محور W مشابه، با L=40 و B* = 1×56×56 = 3136.
'''

'\nپیگیری شکل\u200cها با مثال واقعی\n        فرض ورودی\u200cهای استیج: x_q.shape = x_kv.shape = (1, 32, 56, 56, 40)\n        محور H:\n        q_seq, kv_seq: (B*D*W, H, C) = (2240, 56, 32)، L=56\n        Q,K,V: (2240, 4, 56, 8)\n        logits: (2240, 4, 56, 56) (+bias) → softmax\n        خروجی headها: (2240, 4, 56, 8) → ادغام → (2240, 56, 32)\n        برگشت به 5بعدی: (1, 32, 56, 56, 40)\n        محور W مشابه، با L=40 و B* = 1×56×56 = 3136.\n'

In [ ]:
class CrossModalAxialBlock(nn.Module):
    def __init__(self, dim, num_heads=4, max_len=256,
                 attn_dropout=0.0, proj_dropout=0.0, prenorm=False, use_rel_bias=True):
        super().__init__()
        cfg = dict(dim=dim, num_heads=num_heads, max_len=max_len,
                   attn_dropout=attn_dropout, proj_dropout=proj_dropout,
                   prenorm=prenorm, use_rel_bias=use_rel_bias)
        '''
        یک دیکشنری مشترک از هایپرپارامترها می‌سازد تا تکرار کد کمتر شود.
        dim: تعداد کانال‌ها (C). باید با num_heads سازگار باشد (dim % num_heads == 0).
        max_len: سقف طول توالی برای relative bias (باید ≥ بزرگ‌ترین L در آن استیج باشد)
        '''

        self.d = CrossAxialAttention(axis='D', **cfg)
        # در پایتون، عملگر ** یعنی keyword-argument unpacking.
        # یعنی یک دیکشنری که کلیدهایش اسم پارامترها و مقدارهایش مقادیر آن پارامترها هستند، موقع فراخوانی تابع/کلاس «باز» می‌شود و انگار همان پارامترها را تک‌به‌تک نوشته‌ای.
        self.h = CrossAxialAttention(axis='H', **cfg)
        self.w = CrossAxialAttention(axis='W', **cfg)
        '''
        این بلوک، کراس‌اَتنشن محوری را سه بار و به‌ترتیب روی محورهای D→H→W روی خروجی A اجرا می‌کند؛ هر بار A از B کمک می‌گیرد.
        یعنی: در محور عمق (D) A یاد می‌گیرد «به کجای B نگاه کند»، سپس همین کار روی ارتفاع (H)، و بعد عرض (W).
        '''

    def forward(self, xa, xb):
        # xa = مدالیته A (مثلاً T1) ، xb = مدالیته B (مثلاً T1Gd)
        xa = self.d(xa, xb);
        xa = self.h(xa, xb);
        xa = self.w(xa, xb)
        return xa


In [ ]:
#@title عنوانِ این سلول
#@markdown (اختیاری) توضیح کوتاه اینجاست

'''
شکل‌ها با مثال شما

    فرض دو تنسور هم‌اندازه (از ConvStem):
    xa.shape = xb.shape = (B=1, C=32, D=56, H=56, W=40)

    محور D:
            B* = B×H×W = 2240، L = 56

            Q/K/V: (B*, h, L, d) = (2240, 4, 56, 8)

            بعد از SDPA/softmax و ادغام هدها → برگشت به (1, 32, 56, 56, 40).

            محور H: مشابه با L=56 و B* = B×D×W = 2240.

            محور W: L=40 و B* = B×D×H = 3136.

            در هر گام شکل تغییر نمی‌کند؛ فقط ویژگی A با راهنمایی B به‌روز می‌شود.
            '''

'\nشکل\u200cها با مثال شما\n\n    فرض دو تنسور هم\u200cاندازه (از ConvStem):\n    xa.shape = xb.shape = (B=1, C=32, D=56, H=56, W=40)\n\n    محور D:\n            B* = B×H×W = 2240، L = 56\n\n            Q/K/V: (B*, h, L, d) = (2240, 4, 56, 8)\n\n            بعد از SDPA/softmax و ادغام هدها → برگشت به (1, 32, 56, 56, 40).\n\n            محور H: مشابه با L=56 و B* = B×D×W = 2240.\n\n            محور W: L=40 و B* = B×D×H = 3136.\n\n            در هر گام شکل تغییر نمی\u200cکند؛ فقط ویژگی A با راهنمایی B به\u200cروز می\u200cشود.\n            '

In [ ]:
class MCCABlock(nn.Module):
    def __init__(self, dim, num_heads=4, max_len=256, use_gating=True,
                 attn_dropout: float = 0.0, proj_dropout: float = 0.0,
                 prenorm: bool = False, use_rel_bias: bool = True,
                 ffn_dropout: float = 0.0):
        super().__init__()

        self.self_a = SelfModalAxialBlock(
            dim, num_heads=num_heads, max_len=max_len,
            attn_dropout=attn_dropout, proj_dropout=proj_dropout,
            prenorm_attn=prenorm, use_rel_bias=use_rel_bias,
            use_ffn=True, ffn_dropout=ffn_dropout, ffn_prenorm=False
        )
        self.self_b = SelfModalAxialBlock(
            dim, num_heads=num_heads, max_len=max_len,
            attn_dropout=attn_dropout, proj_dropout=proj_dropout,
            prenorm_attn=prenorm, use_rel_bias=use_rel_bias,
            use_ffn=True, ffn_dropout=ffn_dropout, ffn_prenorm=False
        )

        self.cross_a = CrossModalAxialBlock(
            dim, num_heads=num_heads, max_len=max_len,
            attn_dropout=attn_dropout, proj_dropout=proj_dropout,
            prenorm=prenorm, use_rel_bias=use_rel_bias
        )
        self.cross_b = CrossModalAxialBlock(
            dim, num_heads=num_heads, max_len=max_len,
            attn_dropout=attn_dropout, proj_dropout=proj_dropout,
            prenorm=prenorm, use_rel_bias=use_rel_bias
        )

        self.use_gating = use_gating

        if use_gating:
            self.gate_a = nn.Sequential(nn.Conv3d(dim, dim, 1), nn.Sigmoid())
            self.gate_b = nn.Sequential(nn.Conv3d(dim, dim, 1), nn.Sigmoid())
        '''
        هدف از گیتینگ: به‌صورت تطبیقی و موضعی تصمیم بگیریم کِی اطلاعات مدالیته‌ی مقابل به درد می‌خورد و کِی بهتر است روی خودِ مدالیته تکیه کنیم. این کار جلوی «نویز-لیکیج» را می‌گیرد (مثلاً جایی که T1Gd کمکی نمی‌کند، وارد نکند)
        T1Gd معمولاً ناحیه‌ی enhancing tumor را خوب نشان می‌دهد. پس در همان وکسل‌ها  (برای A=T1) بالا می‌رود تا A_cross را بیشتر وارد کند.
        در نواحی ادیما، FLAIR مفیدتر است؛ برای شاخه‌ی (T2, FLAIR) گیتِ مربوطه روی وکسل‌های ادیما بالا می‌رود تا cross از FLAIR وزن بیشتری بگیرد.   در نواحی نرمال یا نکروز که مدالیته‌ی مقابل کمکی ندارد، گیت پایین می‌ماند و مسیر self غالب می‌شود.
        '''

    def forward(self, xa, xb):
        # Self محوری
        xa_s = self.self_a(xa)
        xb_s = self.self_b(xb)

        # Cross محوری (A از B و B از A)
        xa_c = self.cross_a(xa_s, xb_s)
        # xa_c همان A است که با اطلاعات B تقویت شده.
        xb_c = self.cross_b(xb_s, xa_s)

        if self.use_gating:
            ga = self.gate_a(xa_s)
            # ga تصمیم‌گیر است: کجاهای A واقعاً از B سود می‌برد؟
            gb = self.gate_b(xb_s)

            xa_out = ga * xa_c + (1 - ga) * xa_s
            xb_out = gb * xb_c + (1 - gb) * xb_s
            '''
            فرض کن در یک وکسل خاص:

                    A_self(c) = 0.4
                    A_cross(c) = 1.2 (یعنی B در این وکسل سیگنال قوی‌تری دارد)
                    گیت پیش‌بینی کرده: g_A(c) = 0.8
                                A out (c)=0.8×1.2+0.2×0.4=0.96+0.08=1.04
                                خروجی به سمت cross می‌رود چون آن‌جا مفیدتر است.

                    برعکس، اگر جای دیگری g_A(c)=0.1 و A_cross(c) بی‌ارزش باشد:
                                Aout(c)=0.1×Across​(c)+0.9×Aself​(c)≈Aself​(c)
                                یعنی اطلاعات بی‌فایده از B را عملاً «خاموش» می‌کند.
            '''

        else:
            xa_out = xa_s + xa_c
            xb_out = xb_s + xb_c
        return xa_out, xb_out


In [ ]:
#@title عنوانِ این سلول
#@markdown (اختیاری) توضیح کوتاه اینجاست

'''
# فرض: ConvStem, SelfModalAxialBlock تعریف شده‌اند
device = "cuda" if torch.cuda.is_available() else "cpu"
x_t1   = torch.randn(1, 1, 224, 224, 160, device=device)
x_t1gd = torch.randn(1, 1, 224, 224, 160, device=device)

stem = ConvStem(in_channels=1, base_channels=32).to(device)
t1_l2   = stem(x_t1);   t1_l2   = t1_l2[-1] if isinstance(t1_l2, (tuple,list)) else t1_l2
t1gd_l2 = stem(x_t1gd); t1gd_l2 = t1gd_l2[-1] if isinstance(t1gd_l2, (tuple,list)) else t1gd_l2
print("stem out:", t1_l2.shape)  # (1,32,56,56,40)

# CrossAxialAttention تک‌محوره (نمونه محور H)
xq = t1_l2
xkv = t1gd_l2
attnH = CrossAxialAttention(dim=32, num_heads=4, axis='H', max_len=256).to(device)
with torch.no_grad():
    yH = attnH(xq, xkv)
print("Cross Axial H:", yH.shape)  # (1,32,56,56,40)

# CrossModalAxialBlock سه‌محوره
xq = t1_l2; xkv = t1gd_l2
xq2 = xq.clone()
cblock = CrossModalAxialBlock(dim=32, num_heads=4, max_len=256).to(device)
with torch.no_grad():
    y_cross = cblock(xq, xkv)
print("CrossModalAxialBlock:", y_cross.shape)  # (1,32,56,56,40)

# MCCA (دو خروجی)
mcca = MCCABlock(dim=32, num_heads=4, max_len=256, use_gating=True).to(device)
with torch.no_grad():
    a_out, b_out = mcca(t1_l2, t1gd_l2)
print("MCCA A/B:", a_out.shape, b_out.shape)  # هر دو: (1,32,56,56,40)
'''

'\n# فرض: ConvStem, SelfModalAxialBlock تعریف شده\u200cاند\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\nx_t1   = torch.randn(1, 1, 224, 224, 160, device=device)\nx_t1gd = torch.randn(1, 1, 224, 224, 160, device=device)\n\nstem = ConvStem(in_channels=1, base_channels=32).to(device)\nt1_l2   = stem(x_t1);   t1_l2   = t1_l2[-1] if isinstance(t1_l2, (tuple,list)) else t1_l2\nt1gd_l2 = stem(x_t1gd); t1gd_l2 = t1gd_l2[-1] if isinstance(t1gd_l2, (tuple,list)) else t1gd_l2\nprint("stem out:", t1_l2.shape)  # (1,32,56,56,40)\n\n# CrossAxialAttention تک\u200cمحوره (نمونه محور H)\nxq = t1_l2\nxkv = t1gd_l2\nattnH = CrossAxialAttention(dim=32, num_heads=4, axis=\'H\', max_len=256).to(device)\nwith torch.no_grad():\n    yH = attnH(xq, xkv)\nprint("Cross Axial H:", yH.shape)  # (1,32,56,56,40)\n\n# CrossModalAxialBlock سه\u200cمحوره\nxq = t1_l2; xkv = t1gd_l2\nxq2 = xq.clone()\ncblock = CrossModalAxialBlock(dim=32, num_heads=4, max_len=256).to(device)\nwith torch.no_grad():\n    y_cross

In [ ]:
import torch
import torch.nn as nn

# --- Downsample + Channel Expansion ---
class DownBlock3D(nn.Module):
    """
    Conv3d stride=2 برای نصف‌کردن D/H/W و افزایش کانال‌ها.
    ورودی/خروجی: (B, Cin, D, H, W) → (B, Cout, D/2, H/2, W/2)
    """
    def __init__(self, in_ch, out_ch, norm='bn', negative_slope=0.1):
        super().__init__()
        norm_layer = {
            'bn': nn.BatchNorm3d,
            'in': nn.InstanceNorm3d,
            'gn': lambda c: nn.GroupNorm(num_groups=min(8, c), num_channels=c)
        }['bn']
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, stride=2, padding=1, bias=False),
            norm_layer(out_ch),
            nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class DualBranchEncoder(nn.Module):
    """
    دو شاخه‌ی مدالیته‌ای:
      - Branch A: T1 ↔ T1Gd
      - Branch B: T2 ↔ FLAIR
    در هر لِول (¼, ⅛, 1⁄16) یک MCCA روی جفت مدالیته‌ی همان شاخه می‌زنیم.
    DownBlockها بین لِول‌ها ابعاد فضایی را نصف و کانال‌ها را دوبرابر می‌کنند.

    ورودی‌ها:
      t1, t1gd, t2, flair  با شکل (B, 1, 224, 224, 160)

    خروجی (دیکشنری):
      - 't1','t1gd','t2','flair': لیست فیچرهای [L2, L3, L4]
        L2:  (B, C, 56, 56, 40)
        L3:  (B, 2C, 28, 28, 20)
        L4:  (B, 4C, 14, 14, 10)
      - 'levels': ['L2','L3','L4']  (برای راحتیِ دی‌کدر)
    """

    def __init__(self,
        in_channels: int = 1,
        base_channels: int = 32,
        num_heads: int = 4,
        max_len: int = 256,
        use_gating: bool = True,
        attn_dropout: float = 0.0,
        proj_dropout: float = 0.0,
        prenorm: bool = False,
        ffn_dropout: float = 0.0,   # ← اضافه شد
        ):
        super().__init__()
        self.attn_dropout = attn_dropout
        self.proj_dropout = proj_dropout
        self.ffn_dropout  = ffn_dropout
        self.prenorm      = prenorm
        self.use_gating   = use_gating
        self.num_heads    = num_heads
        self.max_len      = max_len


        C2 = base_channels          # لِول ¼
        C3 = base_channels * 2      # لِول ⅛
        C4 = base_channels * 4      # لِول 1⁄16

        # ----- 4× ConvStem (برای هر مدالیته مستقل) -----
        self.stem_t1    = ConvStem(in_channels, base_channels, return_all=False)
        self.stem_t1gd  = ConvStem(in_channels, base_channels, return_all=False)
        self.stem_t2    = ConvStem(in_channels, base_channels, return_all=False)
        self.stem_flair = ConvStem(in_channels, base_channels, return_all=False)

        # ----- DownBlocks برای هر مدالیته (¼→⅛ و ⅛→1⁄16) -----
        # ¼ → ⅛
        self.down_t1_L2  = DownBlock3D(C2, C3)
        self.down_t1g_L2 = DownBlock3D(C2, C3)
        self.down_t2_L2  = DownBlock3D(C2, C3)
        self.down_fl_L2  = DownBlock3D(C2, C3)
        # ⅛ → 1⁄16
        self.down_t1_L3  = DownBlock3D(C3, C4)
        self.down_t1g_L3 = DownBlock3D(C3, C4)
        self.down_t2_L3  = DownBlock3D(C3, C4)
        self.down_fl_L3  = DownBlock3D(C3, C4)

        # ----- MCCA برای هر شاخه، در هر لِول -----
        mcca_common = dict(
        num_heads=num_heads, max_len=max_len, use_gating=use_gating,
        attn_dropout=attn_dropout, proj_dropout=proj_dropout,
        prenorm=prenorm, use_rel_bias=True,
        ffn_dropout=ffn_dropout,   # ← این خط مهم
        )
        mcca_cfg_L2 = dict(dim=C2, **mcca_common)
        mcca_cfg_L3 = dict(dim=C3, **mcca_common)
        mcca_cfg_L4 = dict(dim=C4, **mcca_common)

        #mcca_cfg_L2 = dict(dim=C2, num_heads=num_heads, max_len=max_len, use_gating=use_gating)
        #mcca_cfg_L3 = dict(dim=C3, num_heads=num_heads, max_len=max_len, use_gating=use_gating)
        #mcca_cfg_L4 = dict(dim=C4, num_heads=num_heads, max_len=max_len, use_gating=use_gating)

        # Branch A (T1 ↔ T1Gd)
        self.mccaA_L2 = MCCABlock(**mcca_cfg_L2)
        self.mccaA_L3 = MCCABlock(**mcca_cfg_L3)
        self.mccaA_L4 = MCCABlock(**mcca_cfg_L4)

        # Branch B (T2 ↔ FLAIR)
        self.mccaB_L2 = MCCABlock(**mcca_cfg_L2)
        self.mccaB_L3 = MCCABlock(**mcca_cfg_L3)
        self.mccaB_L4 = MCCABlock(**mcca_cfg_L4)

    @torch.no_grad()
    def _debug_shapes(self, name, *xs):
        # اگر خواستی شکل‌ها را سریع چک کنی، این را فعال کن:
        # print(name, [tuple(x.shape) for x in xs])
        return

    def forward(self, t1, t1gd, t2, flair):
        # ===== Level-2 (¼ رزولوشن) =====
        t1_L2    = self.stem_t1(t1)          # (B, C2, 56, 56, 40)
        t1g_L2   = self.stem_t1gd(t1gd)
        t2_L2    = self.stem_t2(t2)
        flair_L2 = self.stem_flair(flair)

        self._debug_shapes("L2 stem", t1_L2, t1g_L2, t2_L2, flair_L2)

        # MCCA روی هر شاخه
        t1_L2,  t1g_L2  = self.mccaA_L2(t1_L2,  t1g_L2)    # (B, C2, 56,56,40) و (B, C2, 56,56,40)
        t2_L2,  flair_L2= self.mccaB_L2(t2_L2,  flair_L2)

        # ===== Level-3 (⅛ رزولوشن) =====
        t1_L3    = self.down_t1_L2(t1_L2)                 # (B, C3, 28,28,20)
        t1g_L3   = self.down_t1g_L2(t1g_L2)
        t2_L3    = self.down_t2_L2(t2_L2)
        flair_L3 = self.down_fl_L2(flair_L2)

        self._debug_shapes("L3 down", t1_L3, t1g_L3, t2_L3, flair_L3)

        t1_L3,  t1g_L3   = self.mccaA_L3(t1_L3,  t1g_L3)  # (B, C3, 28,28,20) ×۲
        t2_L3,  flair_L3 = self.mccaB_L3(t2_L3,  flair_L3)

        # ===== Level-4 (1⁄16 رزولوشن) =====
        t1_L4    = self.down_t1_L3(t1_L3)                 # (B, C4, 14,14,10)
        t1g_L4   = self.down_t1g_L3(t1g_L3)
        t2_L4    = self.down_t2_L3(t2_L3)
        flair_L4 = self.down_fl_L3(flair_L3)

        self._debug_shapes("L4 down", t1_L4, t1g_L4, t2_L4, flair_L4)

        t1_L4,  t1g_L4   = self.mccaA_L4(t1_L4,  t1g_L4)  # (B, C4, 14,14,10) ×۲
        t2_L4,  flair_L4 = self.mccaB_L4(t2_L4,  flair_L4)

        # ===== خروجی‌ها برای Skipها و Bottleneck بعدی =====
        out = {
            "levels": ["L2", "L3", "L4"],
            "t1":    [t1_L2,    t1_L3,    t1_L4],
            "t1gd":  [t1g_L2,   t1g_L3,   t1g_L4],
            "t2":    [t2_L2,    t2_L3,    t2_L4],
            "flair": [flair_L2, flair_L3, flair_L4],
        }

        #print("L2 ch:", t1_L2.shape, t1g_L2.shape, t2_L2.shape, flair_L2.shape)
        #print("L3 ch:", t1_L3.shape, t1g_L3.shape, t2_L3.shape, flair_L3.shape)
        #print("L4 ch:", t1_L4.shape, t1g_L4.shape, t2_L4.shape, flair_L4.shape)

        return out


In [ ]:
#@title تست
#@markdown

'''
device = "cuda" if torch.cuda.is_available() else "cpu"
B, C, H, W, D = 1, 1, 224, 224, 160  # برای تست سریع؛ بعداً 224,224,160 بگذار
t1    = image_aug[:, 0:1, :, :]
t1gd  = image_aug[:, 1:2, :, :]
t2    = image_aug[:, 2:3, :, :]
flair = image_aug[:, 3:4, :, :]

enc = DualBranchEncoder(in_channels=1, base_channels=32, num_heads=4, max_len=128).to(device)
with torch.no_grad():
    out = enc(t1, t1gd, t2, flair)

print(out["levels"])
for k in ["t1", "t1gd", "t2", "flair"]:
    print(k, [tuple(x.shape) for x in out[k]])
 '''



'\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\nB, C, H, W, D = 1, 1, 224, 224, 160  # برای تست سریع؛ بعداً 224,224,160 بگذار\nt1    = image_aug[:, 0:1, :, :]\nt1gd  = image_aug[:, 1:2, :, :]\nt2    = image_aug[:, 2:3, :, :]\nflair = image_aug[:, 3:4, :, :]\n\nenc = DualBranchEncoder(in_channels=1, base_channels=32, num_heads=4, max_len=128).to(device)\nwith torch.no_grad():\n    out = enc(t1, t1gd, t2, flair)\n\nprint(out["levels"])\nfor k in ["t1", "t1gd", "t2", "flair"]:\n    print(k, [tuple(x.shape) for x in out[k]])\n '

In [ ]:

#@title 💠 Buttleneck
#@markdown


In [ ]:
import torch
import torch.nn as nn

class BottleneckBlock(nn.Module):
    """
    Bottleneck برای سطح 1/16.
    فرض ورودی هر مدالیته در L4: (B, dim, D, H, W)
      - A شاخه‌ی (T1, T1Gd)
      - B شاخه‌ی (T2, FLAIR)
    طراحی:
      [cat 2*dim → 1×1×1 → dim]  ← برای هر شاخه
      SelfAxial(A), SelfAxial(B)
      CrossAxial(A←B), CrossAxial(B←A)
      Gating اختیاری
      cat(A_out, B_out) → 1×1×1 → dim  ← خروجی نهایی bottleneck
    نکتهٔ مهم: در __init__ این کلاس، dim باید «کانال واقعی L4» باشد (با پروب انکدر به‌دست می‌آید).
    """

    def __init__(self, dim, num_heads=4, max_len=256,
                 use_gating=True, use_self_refine=True,
                 attn_dropout=0.0, proj_dropout=0.0,
                 prenorm=False, use_rel_bias=True,
                 norm_type: str = "in", negative_slope: float = 0.1,
                 ffn_dropout: float = 0.0):
        super().__init__()
        self.use_gating = use_gating
        self.use_self_refine = use_self_refine

        # --- Helper برای نرمال‌سازی ---
        def _norm(ch):
            if norm_type == "bn":   return nn.BatchNorm3d(ch)
            elif norm_type == "gn": return nn.GroupNorm(num_groups=8, num_channels=ch)
            else:                   return nn.InstanceNorm3d(ch, affine=True)  # "in" پیش‌فرض

        # 1) جفت‌سازی مدالیتی‌ها و کاهش به dim (برای هر شاخه)
        self.pairA = nn.Sequential(
            nn.Conv3d(2 * dim, dim, kernel_size=1, bias=False),
            _norm(dim),
            nn.LeakyReLU(negative_slope=negative_slope, inplace=True)
        )
        self.pairB = nn.Sequential(
            nn.Conv3d(2 * dim, dim, kernel_size=1, bias=False),
            _norm(dim),
            nn.LeakyReLU(negative_slope=negative_slope, inplace=True)
        )

        # 2) پالایش درون‌شاخه با Self-Axial
        '''self.selfA = SelfModalAxialBlock(
            dim=dim, num_heads=num_heads, max_len=max_len,
            attn_dropout=attn_dropout, proj_dropout=proj_dropout,
            prenorm_attn=prenorm, use_rel_bias=use_rel_bias,
            use_ffn=False  # سبک
        )
        self.selfB = SelfModalAxialBlock(
            dim=dim, num_heads=num_heads, max_len=max_len,
            attn_dropout=attn_dropout, proj_dropout=proj_dropout,
            prenorm_attn=prenorm, use_rel_bias=use_rel_bias,
            use_ffn=False
        )'''
        self.selfA = SelfModalAxialBlock(
        dim=dim, num_heads=num_heads, max_len=max_len,
        attn_dropout=attn_dropout, proj_dropout=proj_dropout,
        prenorm_attn=prenorm, use_rel_bias=use_rel_bias,
        use_ffn=True, ffn_dropout=ffn_dropout, ffn_prenorm=False
        )
        self.selfB = SelfModalAxialBlock(
            dim=dim, num_heads=num_heads, max_len=max_len,
            attn_dropout=attn_dropout, proj_dropout=proj_dropout,
            prenorm_attn=prenorm, use_rel_bias=use_rel_bias,
            use_ffn=True, ffn_dropout=ffn_dropout, ffn_prenorm=False
        )


        # 3) تعامل بین شاخه‌ها با Cross-Axial
        self.crossA = CrossModalAxialBlock(
            dim=dim, num_heads=num_heads, max_len=max_len,
            attn_dropout=attn_dropout, proj_dropout=proj_dropout,
            prenorm=prenorm, use_rel_bias=use_rel_bias
        )
        self.crossB = CrossModalAxialBlock(
            dim=dim, num_heads=num_heads, max_len=max_len,
            attn_dropout=attn_dropout, proj_dropout=proj_dropout,
            prenorm=prenorm, use_rel_bias=use_rel_bias
        )

        # 4) گیتینگ اختیاری
        if use_gating:
            self.gateA = nn.Sequential(nn.Conv3d(dim, dim, 1, bias=True), nn.Sigmoid())
            self.gateB = nn.Sequential(nn.Conv3d(dim, dim, 1, bias=True), nn.Sigmoid())

        # 5) فیوژن نهایی (فقط cat دو شاخه) → dim
        self.fuse = nn.Conv3d(2 * dim, dim, kernel_size=1, bias=False)

        # init سبک برای 1×1×1 ها
        for m in self.modules():
            if isinstance(m, nn.Conv3d) and m.kernel_size == (1,1,1):
                nn.init.kaiming_uniform_(m.weight, a=1e-2)

    def forward(self, t1_L4, t1gd_L4, t2_L4, fl_L4):
        # ورودی‌ها: هرکدام (B, dim, D, H, W)
        # 1) جفت‌سازی و کاهش برای هر شاخه
        A0 = self.pairA(torch.cat([t1_L4,  t1gd_L4], dim=1))  # (B, dim, ...)
        B0 = self.pairB(torch.cat([t2_L4,  fl_L4],   dim=1))  # (B, dim, ...)

        # 2) Self-Axial (اختیاری self_refine روشن است)
        A_s = self.selfA(A0) if self.use_self_refine else A0
        B_s = self.selfB(B0) if self.use_self_refine else B0

        # 3) Cross-Axial (تعامل A↔B)
        A_c = self.crossA(A_s, B_s)
        B_c = self.crossB(B_s, A_s)

        # 4) گیت یا رزیدوال ساده
        if self.use_gating:
            gA = self.gateA(A_s)
            gB = self.gateB(B_s)
            A_out = gA * A_c + (1.0 - gA) * A_s
            B_out = gB * B_c + (1.0 - gB) * B_s
        else:
            A_out = A_s + A_c
            B_out = B_s + B_c

        # 5) فیوژن نهایی
        fused_in = torch.cat([A_out, B_out], dim=1)  # (B, 2*dim, ...)
        # ایمنی: تنها همین دو را cat می‌کنیم؛ نه چیز دیگر
        # assert fused_in.shape[1] == 2*dim, f"Expected {2*dim} ch, got {fused_in.shape[1]}"
        fused = self.fuse(fused_in)                  # (B, dim, ...)
        return fused, A_out, B_out


In [ ]:
#@title skip this module
#@markdown


import torch
import torch.nn as nn

# فرض: این‌ها را قبلاً داری
# from your_file import SelfModalAxialBlock, CrossModalAxialBlock

class BottleneckBlock_prev(nn.Module):
    """
    Bottleneck Fusion at 1/16 resolution.
    ورودی‌ها: t1_L4, t1gd_L4, t2_L4, flair_L4  (هرکدام: B, C4, D, H, W)
    اگر base=32 باشد، معمولاً C4=128 و برای ورودی 224×224×160، لِول 1/16 تقریباً D,H,W ≈ 10,14,14.
    خروجی‌ها:
      - fused: فیچر نهایی برای دیکودر (B, C4, D, H, W)
              یک فیچر نهایی واحد برای ورود به دیکودر (ترجیحاً هم‌کانال با dim)
      - a_out: شاخه A پس از کراس+گیت (B, C4, D, H, W)
      - b_out: شاخه B پس از کراس+گیت (B, C4, D, H, W)
              a_out و b_out: دو شاخه‌ی پالایش‌شده (برای اسکیپ یا تحلیل)
    """
    def __init__(
        self,
        dim: int,                 # C4 (مثلاً 128 وقتی base=32)
        num_heads: int = 4,
        max_len: int = 256,
        use_gating: bool = True,
        use_self_refine: bool = True,     # SelfAxial روی هر شاخه قبل از کراس
        attn_dropout: float = 0.0,
        proj_dropout: float = 0.0,
        negative_slope: float = 0.1,
        prenorm: bool = False,
        use_rel_bias: bool = True,
    ):
        super().__init__()

        # 1) ادغام هر جفت مدالیته داخل شاخه (Concat -> Conv1x1)
        self.fuse_pair_A = nn.Sequential(
            nn.Conv3d(2 * dim, dim, kernel_size=1, bias=False),
            nn.BatchNorm3d(dim),
            nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
        )
        self.fuse_pair_B = nn.Sequential(
            nn.Conv3d(2 * dim, dim, kernel_size=1, bias=False),
            nn.BatchNorm3d(dim),
            nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
        )

        '''
        T1 و T1Gd (یا T2 و FLAIR در fuse_pair_B) اطلاعات هم‌خانواده‌اند؛ یک فیوژن سریع کانالی با BN و فعال‌ساز برای «یک نمای شاخه‌ای» می‌سازیم:
        a = fuse_pair_A([T1,T1Gd]) و b = fuse_pair_B([T2,FLAIR]).
        نکته: Conv1×1×1 مثل یک Linear per-voxel عمل می‌کند؛ spatial receptive field اضافه نمی‌کند، اما ترکیب یادگرفتنی کانال‌ها را می‌دهد.
        '''

        # 2) اختیاری: پالایش درون‌مدالیته‌ای هر شاخه با Self Axial (D→H→W)
        self.use_self_refine = use_self_refine
        if use_self_refine:
            # روی a و b جداگانه، Self-Axial (D→H→W) می‌زنیم تا هر شاخه کانتکست دوربرد 3D بگیرد و فیچرهاش منسجم شود.
            # قبل از این‌که شاخه A از شاخه B سرنخ بگیرد (کراس)، بهتر است خودش «مرتب» شود تا کراس اشتباه (mis-attend) کمتر شود.
            self.selfA = SelfModalAxialBlock(dim=dim, num_heads=num_heads, max_len=max_len,
                                 attn_dropout=attn_dropout, proj_dropout=proj_dropout,
                                 prenorm_attn=prenorm, use_rel_bias=use_rel_bias)
            self.selfB = SelfModalAxialBlock(dim=dim, num_heads=num_heads, max_len=max_len,
                                 attn_dropout=attn_dropout, proj_dropout=proj_dropout,
                                 prenorm_attn=prenorm, use_rel_bias=use_rel_bias)



        # 3) کراس‌اَتنشن محوری بین دو شاخه (A ← B و B ← A)
            '''
            این‌جا «فیوژن بین‌خانواده‌ای» رخ می‌دهد (مثلاً ویژگی‌های core از شاخه A با ویژگی‌های edema از شاخه B هماهنگ می‌شوند).
            مزیت نسبت به concat: هم پوزیشن-آگاه است، هم تطبیقی: می‌فهمد «کجا از B» برای «کجای A» مفید است.
            '''
        self.crossA = CrossModalAxialBlock(dim=dim, num_heads=num_heads, max_len=max_len,
                                           attn_dropout=attn_dropout, proj_dropout=proj_dropout,
                                           prenorm=prenorm, use_rel_bias=use_rel_bias)
        self.crossB = CrossModalAxialBlock(dim=dim, num_heads=num_heads, max_len=max_len,
                                           attn_dropout=attn_dropout, proj_dropout=proj_dropout,
                                           prenorm=prenorm, use_rel_bias=use_rel_bias)

        # 4) گیت برای فیوژن تطبیقی cross vs self در هر شاخه
        # در هر وکسل/کانال تصمیم بگیرد چه‌قدر از cross وارد شود و چه‌قدر از self بماند. کنترل نویز/لیک و تطابق موضعی دقیق.
        self.use_gating = use_gating
        if use_gating:
            self.gateA = nn.Sequential(nn.Conv3d(dim, dim, kernel_size=1), nn.Sigmoid())
            self.gateB = nn.Sequential(nn.Conv3d(dim, dim, kernel_size=1), nn.Sigmoid())

        # 5) ادغام نهایی دو شاخه به یک فیچر (Concat -> Conv1x1)
        # پس از گیتینگ، concat([a_out, b_out], dim=1) و یک Conv1×1×1 تا یک فیچر واحد برای دیکودر بسازیم.
        self.fuse_out = nn.Sequential(
            nn.Conv3d(2 * dim, dim, kernel_size=1, bias=False),
            nn.BatchNorm3d(dim),
            nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
        )

    def forward(self, t1_L4, t1gd_L4, t2_L4, flair_L4):
        # ----- Pair fusion داخل هر شاخه -----
        # Branch A: (T1, T1Gd)
        a = self.fuse_pair_A(torch.cat([t1_L4, t1gd_L4], dim=1))   # (B, C, D,H,W)
        # Branch B: (T2, FLAIR)
        b = self.fuse_pair_B(torch.cat([t2_L4, flair_L4], dim=1))  # (B, C, D,H,W)

        # ----- Self refinement اختیاری -----
        if self.use_self_refine:
            a_s = self.selfA(a)
            b_s = self.selfB(b)
        else:
            a_s, b_s = a, b

        # ----- Cross between branches -----
        a_c = self.crossA(a_s, b_s)   # A از B
        b_c = self.crossB(b_s, a_s)   # B از A

        # ----- Gated fusion درون شاخه -----
        if self.use_gating:
            ga = self.gateA(a_s)
            gb = self.gateB(b_s)
            a_out = ga * a_c + (1 - ga) * a_s
            b_out = gb * b_c + (1 - gb) * b_s
        else:
            a_out = a_s + a_c
            b_out = b_s + b_c
        '''
        اگر گیت روشن باشد:
                جایی که cross مفیدتر است، ga بالا می‌رود → وزن cross بیشتر.
                جایی که مفید نیست، ga پایین می‌آید → به self تکیه می‌کنیم.
        اگر خاموش باشد:
                جمع ساده (ارزان‌تر، تطبیق‌پذیری کمتر)
        '''

        # ----- خروجی واحد برای دیکودر -----
        fused = self.fuse_out(torch.cat([a_out, b_out], dim=1))  # (B, C, D,H,W)
        return fused, a_out, b_out


In [ ]:
#@title عنوانِ این سلول
#@markdown (اختیاری) توضیح کوتاه اینجاست

''' در Bottleneck چون D,H,W کوچک‌اند (مثلاً 10,14,14)، AxialAttention هزینه‌ی معقولی دارد ولی اثر سراسری تولید می‌کند.

اگر کمبود حافظه داشتی
use_self_refine=False، یا Cross فقط روی یک/دو محور (مثلاً H یا D,H)، یا num_heads=2 و dim کمتر.

گیت را نرم شروع کن: بایاسِ Conv1×1 در گیت را صفر یا کمی منفی بگذار تا اول کار به self متکی باشد و به‌تدریج cross را بیشتر کند. '''

''' چرا دوباره Self/Cross/Gate در Bottleneck، وقتی قبلاً در MCCAِ انکدر داشتیم؟

مالتی‌اسکیل ≠ تکرار

MCCAهایی که در انکدر گذاشتی در سطوح L2/L3 (رزولوشن بالاتر) کار می‌کنند؛ آن‌جا تعامل‌ها بیشتر محلی/میان‌بُرداری هستند. در Bottleneck (L4, 1/16) نقشه‌ها کوچک و کانال‌ها زیادند؛ همان اَکشنِ محوری (D/H/W) این‌بار معنای کانتکست سراسری‌تر دارد (یک ستون/ردیف تقریباً بخش بزرگی از مغز را می‌بیند). نتیجه: Self در این مقیاس، پالایش بلندبُرد می‌دهد و Cross این‌جا هم‌ترازسازی جهانی A↔B را یاد می‌گیرد—چیزی که L2/L3 به‌دلیل اندازهٔ دنباله و نویز بافت‌ها به این کیفیت نمی‌دهد.

نوع تعامل فرق می‌کند

MCCA انکدر را روی زوج‌های هم‌خانواده گذاشتی: (T1 ↔ T1Gd) و (T2 ↔ FLAIR). در Bottleneck که من پیشنهاد کردم، بعد از ادغام داخل هر زوج، تعامل بین دو شاخه انجام می‌شود: Branch A ↔ Branch B. یعنی این‌جا Cross، فیوژن بین‌خانواده‌ای توموری را یاد می‌گیرد: core/ET (هدایت‌شده از T1Gd/T1) در نسبت با edema (هدایت‌شده از FLAIR/T2). این سطح از هم‌افزایی را MCCAهای بالادستی پوشش نمی‌دهند.

توزیع ویژگی بعد از داون‌سمپل عوض می‌شود

وقتی از L3 → L4 می‌روی، هم چگالی اطلاعات فضایی عوض می‌شود، هم معنای کانال‌ها. یک Self کوتاه در Bottleneck قبل از Cross، فیچرها را بازکالیبره/هموار می‌کند تا Cross-branch اشتباه نرود (mis-attend). این یک «پالایش قبل از ادغام نهایی» است، نه تکرار بی‌هدف.

گِیت در انتهای فیوژن نقش ضدنویز دارد

در پایین‌ترین رزولوشن، خطای یک ادغامِ بد، به کلِ دی‌کدر می‌رسد. گِیت اجازه می‌دهد موضعی تصمیم بگیری: هر جا cross کمک کرد ⇒ وزن بالا، هر جا نه ⇒ self غالب. این ترم ایمنی، انتهای انکدر ضروری‌تر از بالادست است.

''' ''' Self- & Cross-Axial Attention چه کاری اضافه می‌کنند؟ SelfModalAxial: داخل هر شاخه (مثلاً فیوژن T1/T1Gd) وابستگی‌های دوربرد 3D را محور-به-محور (D→H→W) مدل می‌کند. یعنی قبل از تعامل با شاخه دیگر، فیچر خود شاخه «سازمان‌دهی» و «پالایش» می‌شود. CrossModalAxial: برای هر محور، Q از شاخه A و K,V از شاخه B می‌گیرد؛ بنابراین یاد می‌گیرد «برای هر وکسل A، کجاهای B مفیدند». این دقیقاً همان هم‌ترازسازی پویا A↔B است که concat نمی‌تواند انجام دهد. '''

''' چرا در باتلنک «Self» هم لازم است؟ ورودی باتلنک (1/16) کم‌رزولوشن ولی پُرکانال است؛ قبل از اینکه بین شاخه‌ها cross بزنی، بهتر است هر شاخه درون خودش منسجم شود (long-range aggregation). اگر Self را حذف کنی، Cross دارد روی فیچرهای خام‌تر/پخش‌تر شمارش می‌کند و ممکن است mis-attend کند (کجاهای اشتباه B را ببیند). تجربه‌ی عملی (به‌خصوص در Multi-Modal MRI): Self→Cross معمولاً پایدارتر و دقیق‌تر از Crossِ مستقیم است، چون Self «زمینه» را مرتب می‌کند. اگر منابع محدود داری، می‌توانی Self را فقط یک‌بار (نه سه محور) یا با نسخه‌ی لایت (Conv3×3 رزیدوال) جایگزین کنی. ولی «کاملاً» حذف‌کردن، معمولاً کمی دقت را کم می‌کند. '''

''' اگر بخواهی «سبک‌تر» کنی (گزینه‌های لایت) کم‌کردن محورها: فقط روی دو محور (مثلاً D و H) Cross بزن. کاهش headها: num_heads=2 یا dim کوچک‌تر (مثلاً base=24 → C4=96). Self لایت: به‌جای SelfAxial کامل، یک Residual(Conv3×3×3) یا فقط محور H را نگه دار. Cross لایت: فقط یک محور (H) کراس؛ یا از proj_dropout و attn_dropout استفاده کن. گیت لایت: Softmax دونالئی یا حتی کانال‌ثابت (خیلی ارزان، ولی تطبیق فضایی کمتر). حذف Self در باتلنک: قابل‌انجام است؛ اگر OOM داشتی اول این را خاموش کن، ولی انتظار داشته باش کمی دقت بیاید پایین. '''

' اگر بخواهی «سبک\u200cتر» کنی (گزینه\u200cهای لایت) کم\u200cکردن محورها: فقط روی دو محور (مثلاً D و H) Cross بزن. کاهش headها: num_heads=2 یا dim کوچک\u200cتر (مثلاً base=24 → C4=96). Self لایت: به\u200cجای SelfAxial کامل، یک Residual(Conv3×3×3) یا فقط محور H را نگه دار. Cross لایت: فقط یک محور (H) کراس؛ یا از proj_dropout و attn_dropout استفاده کن. گیت لایت: Softmax دونالئی یا حتی کانال\u200cثابت (خیلی ارزان، ولی تطبیق فضایی کمتر). حذف Self در باتلنک: قابل\u200cانجام است؛ اگر OOM داشتی اول این را خاموش کن، ولی انتظار داشته باش کمی دقت بیاید پایین. '

In [ ]:
#@title تست
#@markdown

'''
device = "cuda" if torch.cuda.is_available() else "cpu"

# ابعاد لِول 1/16 نمونه (برای ورودی 224×224×160 و 2 بار /2): 14×14×10
B, C4, D, H, W = 1, 128, 10, 14, 14   # اگر base=32 → C4=128

t1_L4    = torch.randn(B, C4, D, H, W, device=device)
t1gd_L4  = torch.randn(B, C4, D, H, W, device=device)
t2_L4    = torch.randn(B, C4, D, H, W, device=device)
flair_L4 = torch.randn(B, C4, D, H, W, device=device)

bottleneck = BottleneckBlock(
    dim=C4, num_heads=4, max_len=128, use_gating=True,
    prenorm=False, use_self_refine=True ).to(device)

with torch.no_grad():
      fused, a_out, b_out = bottleneck(t1_L4, t1gd_L4, t2_L4, flair_L4)

print("fused :", tuple(fused.shape))   # انتظار: (1, 128, 10, 14, 14)  ← ترتیب D,H,W یا H,W,D بر اساس کدت
print("a_out :", tuple(a_out.shape))   # (1, 128, 10, 14, 14)
print("b_out :", tuple(b_out.shape))   # (1, 128, 10, 14, 14)
'''

'\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\n\n# ابعاد لِول 1/16 نمونه (برای ورودی 224×224×160 و 2 بار /2): 14×14×10\nB, C4, D, H, W = 1, 128, 10, 14, 14   # اگر base=32 → C4=128\n\nt1_L4    = torch.randn(B, C4, D, H, W, device=device)\nt1gd_L4  = torch.randn(B, C4, D, H, W, device=device)\nt2_L4    = torch.randn(B, C4, D, H, W, device=device)\nflair_L4 = torch.randn(B, C4, D, H, W, device=device)\n\nbottleneck = BottleneckBlock(\n    dim=C4, num_heads=4, max_len=128, use_gating=True,\n    prenorm=False, use_self_refine=True ).to(device)\n\nwith torch.no_grad():\n      fused, a_out, b_out = bottleneck(t1_L4, t1gd_L4, t2_L4, flair_L4)\n\nprint("fused :", tuple(fused.shape))   # انتظار: (1, 128, 10, 14, 14)  ← ترتیب D,H,W یا H,W,D بر اساس کدت\nprint("a_out :", tuple(a_out.shape))   # (1, 128, 10, 14, 14)\nprint("b_out :", tuple(b_out.shape))   # (1, 128, 10, 14, 14)\n'

In [ ]:
#@title 💠 Decoder Blocck
#@markdown


In [ ]:
#@title عنوانِ این سلول
#@markdown (اختیاری) توضیح کوتاه اینجاست

'''
UpsampleBlock(in_ch, out_ch, scale_factor=2, mode='trilinear')

SkipGate(ch) (Attention Gate سبک)

SkipProj(in_ch, out_ch) (Conv1×1×1)

TCFCBlock(ch) (هم‌ترازی/فیوژن محوری + گیت داخلی)

DecoderBlock(in_ch, out_ch, use_residual=True)

DecoderStage(in_ch, out_ch, has_skip: bool) ← داخلش: Upsample → (AG+Proj+TCFC اگر has_skip) → DecoderBlock

SegHead(in_ch, num_classes)

AuxHead(in_ch, num_classes) (فقط زمان Train)
'''
'''
ماژول‌هایی که باید بنویسی/بسازی

UpsampleBlock × 4

        Stage-1: 1/16→1/8

        Stage-2: 1/8→1/4

        Stage-3: 1/4→1/2

        Stage-4: 1/2→1×

SkipGate(AG) × 2 (فقط جایی که اسکیپ داریم)

        روی Skip@1/8 و Skip@1/4

SkipProj(1×1×1) × 2

        تطبیق کانال اسکیپ‌ها (1/8 و 1/4) با جریان اصلی

TCFCBlock × 2

        Stage-1: ادغام با Skip@1/8

        Stage-2: ادغام با Skip@1/4

        (Stage-3/4 اسکیپ ندارند در نسخهٔ مینیمال)

DecoderBlock × 4

        یکی بعد از هر استیج (پس از Upsample و TCFC)

SegHead(1×1×1) × 1

        سر نهایی برای logits

AuxHead(Deep Supervision) × 2 (اختیاری ولی توصیه‌شده)

        از خروجی Stage-1@1/8 و Stage-2@1/4

        در آموزش به فول‌رزولوشن upsample و در لا‌س وزن‌دار شرکت می‌کنند

جمع: 4 Upsample + 2 Gate + 2 SkipProj + 2 TCFC + 4 DecoderBlock + 1 SegHead + 2 AuxHead(اختیاری)
'''

'\nماژول\u200cهایی که باید بنویسی/بسازی\n\nUpsampleBlock × 4\n\n        Stage-1: 1/16→1/8\n\n        Stage-2: 1/8→1/4\n\n        Stage-3: 1/4→1/2\n\n        Stage-4: 1/2→1×\n\nSkipGate(AG) × 2 (فقط جایی که اسکیپ داریم)\n\n        روی Skip@1/8 و Skip@1/4\n\nSkipProj(1×1×1) × 2\n\n        تطبیق کانال اسکیپ\u200cها (1/8 و 1/4) با جریان اصلی\n\nTCFCBlock × 2\n\n        Stage-1: ادغام با Skip@1/8\n\n        Stage-2: ادغام با Skip@1/4\n\n        (Stage-3/4 اسکیپ ندارند در نسخهٔ مینیمال)\n\nDecoderBlock × 4\n\n        یکی بعد از هر استیج (پس از Upsample و TCFC)\n\nSegHead(1×1×1) × 1\n\n        سر نهایی برای logits\n\nAuxHead(Deep Supervision) × 2 (اختیاری ولی توصیه\u200cشده)\n\n        از خروجی Stage-1@1/8 و Stage-2@1/4\n\n        در آموزش به فول\u200cرزولوشن upsample و در لا\u200cس وزن\u200cدار شرکت می\u200cکنند\n\nجمع: 4 Upsample + 2 Gate + 2 SkipProj + 2 TCFC + 4 DecoderBlock + 1 SegHead + 2 AuxHead(اختیاری)\n'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

# --- کمک‌تابع نرمال‌سازی ---
def _norm3d(ch: int, norm_type: str = "in") -> nn.Module:
    if norm_type == "bn":
        return nn.BatchNorm3d(ch)
    if norm_type == "gn":
        return nn.GroupNorm(num_groups=8, num_channels=ch)
    return nn.InstanceNorm3d(ch, affine=True)  # "in" پیش‌فرض


# =========================
#      UpsampleBlock
# =========================
class UpsampleBlock(nn.Module):
    """
    بالا-نمونه‌گیری 3D با دو حالت:
      - method="interp":  F.interpolate  +  Conv1×1×1 (پراجکشن) + Norm + Act (+Dropout اختیاری)
      - method="convtrans": ConvTranspose3d(in→out, k=stride=scale) + Norm + Act (+Dropout اختیاری)

    نکته‌ها:
      - اگر در forward، 'target_size=(D,H,W)' بدهید، با روش interpolate دقیقاً به همان ابعاد می‌رسیم
        (در حالت convtrans هم پس از ترنسپوز کانولوشن، یک interpolate سبک می‌زنیم تا سایز دقیق شود).
      - اگر target_size ندهید، از scale_factor استفاده می‌شود.
      - پس از upsample، کانال‌ها به out_ch نگاشت می‌شود (در interp با Conv1×1×1؛ در convtrans مستقیم).
    """

    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        method: str = "interp",                 # "interp" یا "convtrans"
        scale: Tuple[int, int, int] = (2, 2, 2),
        align_corners: bool = False,
        norm_type: str = "in",
        negative_slope: float = 0.1,
        dropout: float = 0.0,
    ):
        super().__init__()
        assert method in {"interp", "convtrans"}

        self.method = method
        self.scale = tuple(scale)
        self.align_corners = align_corners
        self.negative_slope = negative_slope
        self.dropout = nn.Dropout3d(dropout) if dropout and dropout > 0.0 else nn.Identity()

        if method == "interp":
            # upsample → 1×1×1 proj → norm → act
            self.proj = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False),
                _norm3d(out_ch, norm_type),
                nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
            )
        else:
            # ConvTranspose3d(in→out, k=stride=scale)
            self.tconv = nn.ConvTranspose3d(
                in_ch, out_ch, kernel_size=self.scale, stride=self.scale, bias=False
            )
            self.post = nn.Sequential(
                _norm3d(out_ch, norm_type),
                nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
            )

        # init سبک
        for m in self.modules():
            if isinstance(m, nn.Conv3d) and m.kernel_size == (1, 1, 1):
                nn.init.kaiming_uniform_(m.weight, a=1e-2)
            if isinstance(m, nn.ConvTranspose3d):
                nn.init.kaiming_uniform_(m.weight, a=1e-2)

    def forward(self, x: torch.Tensor, target_size: Optional[Tuple[int, int, int]] = None) -> torch.Tensor:
        """
        x: (B, C, D, H, W)
        target_size: اگر بدهید، خروجی دقیقاً به (D_t, H_t, W_t) می‌رسد.
        """
        if self.method == "interp":
            if target_size is None:
                y = F.interpolate(
                    x, scale_factor=self.scale, mode="trilinear", align_corners=self.align_corners
                )
            else:
                y = F.interpolate(
                    x, size=target_size, mode="trilinear", align_corners=self.align_corners
                )
            y = self.proj(y)         # 1×1×1 proj به out_ch + norm + act
            y = self.dropout(y)
            return y
        else:
            # convtranspose
            y = self.tconv(x)
            y = self.post(y)
            # اگر هدف دقیق بدهند، با یک interpolate سبک تنظیم نهایی می‌کنیم
            if target_size is not None and y.shape[-3:] != target_size:
                y = F.interpolate(y, size=target_size, mode="trilinear", align_corners=self.align_corners)
            y = self.dropout(y)
            return y


In [ ]:
#@title تست
#@markdown

'''
# ورودی نمونه: Bottleneck level ~ (B, C4, D, H, W) = (1, 128, 10, 14, 14)
x = torch.randn(1, 128, 10, 14, 14)

blk_interp = UpsampleBlock(128, 64, method="interp", scale_factor=2)
y1 = blk_interp(x)
print("interp:", y1.shape)   # → torch.Size([1, 64, 20, 28, 28])

blk_trans = UpsampleBlock(128, 64, method="transpose")
y2 = blk_trans(x)
print("transpose:", y2.shape)  # → torch.Size([1, 64, 20, 28, 28])
'''

'\n# ورودی نمونه: Bottleneck level ~ (B, C4, D, H, W) = (1, 128, 10, 14, 14)\nx = torch.randn(1, 128, 10, 14, 14)\n\nblk_interp = UpsampleBlock(128, 64, method="interp", scale_factor=2)\ny1 = blk_interp(x)\nprint("interp:", y1.shape)   # → torch.Size([1, 64, 20, 28, 28])\n\nblk_trans = UpsampleBlock(128, 64, method="transpose")\ny2 = blk_trans(x)\nprint("transpose:", y2.shape)  # → torch.Size([1, 64, 20, 28, 28])\n'

In [ ]:
# برای تمیزکردن اسکیپ انکدر
# منطق: مثل Attention U-Net، یک «سیگنال راهنما» از دیکودر (g) و یک «نقشهٔ اسکیپ» از انکدر (x) می‌گیریم؛ هر دو را به یک فضای میانی فشرده نگاشت می‌کنیم، جمع + ReLU، سپس 1×1×1 → Sigmoid تا یک ماسک فضایی α(D,H,W) بسازیم و اسکیپ را فیلتر کنیم: x_gated = α ⊙ x.

class SkipGate(nn.Module):
    """
    گیت‌کردن اسکیپ با توجه به فیچر دیکودر:
      ورودی‌ها:
        x:  فیچرِ دیکودر در همان رزولوشن (B, Cx, D, H, W)
        s:  اسکیپِ انکدر در همان رزولوشن (B, Cs, D, H, W)
      خروجی:
        s_gated: اسکیپ فیلترشده (B, Cs, D, H, W)

    ایده:
      - از x و s، پروجکشن‌های 1×1×1 به فضای مشترک inter_ch می‌گیریم و جمع می‌کنیم → ReLU → نقشه‌ی توجه مکانی α
        با Conv( inter_ch → 1 ) و Sigmoid.
      - (اختیاری) گیت کانالی با GAP + MLP (Squeeze-Excitation سبک) روی s.
      - s_gated = s * α_spatial * α_channel  (broadcast)
      - اگر شکل فضایی x و s یکی نبود، x را به سایز s ریسایز می‌کنیم (عموماً x از Upsample آمده و برابر است).
    """

    def __init__(
        self,
        x_ch: int,
        s_ch: int,
        inter_ch: Optional[int] = None,     # اگر None → inter_ch = max(8, min(x_ch, s_ch)//2)
        use_channel_gate: bool = True,
        norm_type: str = "in",
        negative_slope: float = 0.1,
    ):
        super().__init__()
        if inter_ch is None:
            inter_ch = max(8, min(x_ch, s_ch) // 2)

        # پروجکشن‌های 1×1×1 به فضای مشترک
        self.qx = nn.Conv3d(x_ch, inter_ch, kernel_size=1, bias=False)
        self.ks = nn.Conv3d(s_ch, inter_ch, kernel_size=1, bias=False)
        self.bn = _norm3d(inter_ch, norm_type)
        self.act = nn.LeakyReLU(negative_slope=negative_slope, inplace=True)

        # نقشه‌ی توجه مکانی: inter_ch → 1
        self.alpha_sp = nn.Sequential(
            nn.Conv3d(inter_ch, 1, kernel_size=1, bias=True),
            nn.Sigmoid()
        )

        # گیت کانالی اختیاری (SE سبک روی s)
        self.use_channel_gate = use_channel_gate
        if use_channel_gate:
            r = max(1, s_ch // 8)
            self.se = nn.Sequential(
                nn.AdaptiveAvgPool3d(1),                 # (B, Cs, 1,1,1)
                nn.Conv3d(s_ch, r, kernel_size=1, bias=True),
                nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
                nn.Conv3d(r, s_ch, kernel_size=1, bias=True),
                nn.Sigmoid()
            )

        # init سبک
        for m in self.modules():
            if isinstance(m, nn.Conv3d) and m.kernel_size == (1, 1, 1):
                nn.init.kaiming_uniform_(m.weight, a=1e-2)

    def forward(self, x: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
        """
        x: (B, Cx, D, H, W)  ← فیچر دیکودر (پس از Upsample)
        s: (B, Cs, D, H, W)  ← اسکیپ انکدر
        """
        # هم‌ترازی فضایی (در صورت نیاز x → شکل s)
        if x.shape[-3:] != s.shape[-3:]:
            x = F.interpolate(x, size=s.shape[-3:], mode="trilinear", align_corners=False)

        # پروجکشن‌ها و ساخت α_spatial
        q = self.qx(x)                # (B, inter, D,H,W)
        k = self.ks(s)                # (B, inter, D,H,W)
        z = self.act(self.bn(q + k))  # fuse → norm → relu
        alpha = self.alpha_sp(z)      # (B,1,D,H,W), 0..1

        # گیت کانالی اختیاری
        if self.use_channel_gate:
            alpha_ch = self.se(s)     # (B, Cs, 1,1,1)
            s = s * alpha_ch

        # اعمال گیت فضایی
        s_gated = s * alpha           # broadcast روی کانال‌های s
        return s_gated

In [ ]:
#@title عنوانِ این سلول
#@markdown (اختیاری) توضیح کوتاه اینجاست

'''
در هر استیج دیکودر:
       1- اول با UpsampleBlock ابعاد فضایی بزرگ‌تر می‌شود (مثلاً از 1/8 به 1/4).
       2- خروجی همین Upsample را داریم؛ به آن می‌گوییم g (فیچر راهنمای دیکودر در این سطح).
       3- هم‌زمان، از انکدر یک اسکیپِ همان سطح داریم؛ به آن می‌گوییم x (فیچر خامِ انکدر در همان رزولوشن).

شکل‌ها (نمونه):
        g: (B, C_dec, D, H, W) ← خروجی Upsample همین استیج
        x: (B, C_skip, D, H, W) ← اسکیپ از انکدر
        D,H,W در هر دو یکی است (چون در یک رزولوشن هستیم)، فقط تعداد کانال‌ها ممکن است فرق کند.

هدف: اسکیپِ انکدر (x) همه‌جایش مفید نیست؛ ممکن است نویز و بافت‌های بی‌ربط داشته باشد.
SkipGate یک نقشهٔ «اجازه عبور» می‌سازد (یک ماسک فضایی به اسم α بین 0 و 1) که می‌گوید:
«در کدام نقاط از فضا، اسکیپ را زیاد عبور بدهیم و کجاها کم؟»
این تصمیم را با راهنمایی g می‌گیرد؛ یعنی خود دیکودر (g) تعیین می‌کند الان کجای تصویر به چه اطلاعاتی نیاز دارد، و بر همان اساس اسکیپ فیلتر می‌شود.

x = اسکیپِ انکدر در همان رزولوشن (مثلاً Skip@1/4).

g = خروجی دیکودر پس از Upsample همین استیج (همان سطح رزولوشن: 1/4).

داخل SkipGate چه اتفاقی رخ می‌دهد؟ (ساده و تصویری)

        فرض کن g مثل «لیست نیازهای دیکودر» در این سطح است و x «جعبهٔ ابزار» انکدر. SkipGate با کمک g تصمیم می‌گیرد از x کجا و چقدر استفاده کند.

        خلاصه‌سازیِ x و g:
        هرکدام با یک کانولوشن 1×1×1 به یک فضای میانی کوچک نگاشت می‌شوند (برای کم‌هزینه و قابل‌جمع شدن شدن).
        (فقط کانال‌ها کوچک می‌شود؛ D,H,W همان می‌ماند.)

        ترکیب سرنخ‌ها:
        این دو خلاصه را جمع می‌کند؛ اگر نقطه‌ای در فضا هم در x قوی باشد و هم g آن را مهم بداند، جمع بزرگ می‌شود.

        ساخت ماسک α:
        روی جمع، نرمال‌سازی و فعال‌ساز می‌گذارد، بعد با یک 1×1×1 خروجی را به یک کانال می‌فشارد و با سیگموید آن را به [0,1] می‌برد.
        نتیجه: α(D,H,W)، یک نقشهٔ اجازه عبور.

        فیلتر اسکیپ:
        اسکیپ فیلتر می‌شود: x_gated = α ⊙ x.
        یعنی در جاهایی که α≈1 است، x تقریباً کامل عبور می‌کند؛ جایی که α≈0 است، x تقریباً قطع می‌شود؛ و بین این دو، عبور نسبی.
        ضرب عنصر به عنصر انجام می‌شود: x_gated = α ⊙ x
        α شکل (B, 1, D, H, W) دارد و روی محور کانال‌ها broadcast می‌شود؛ یعنی برای همهٔ کانال‌های x در یک وکسل، یک α مشترک اعمال می‌گردد.

خروجی SkipGate کجاست و بعدش چه می‌شود؟

        خروجی SkipGate: x_gated با همان شکل و کانال‌های x → (B, C_skip, D, H, W).
        معمولاً بلافاصله یک SkipProj(1×1×1) می‌آید تا کانال‌های x_gated را به C_dec (کانال جریان دیکودر) تبدیل کند.
        سپس این اسکیپِ تمیز و هم‌کانال با خروجی دیکودر (g) وارد TCFCBlock می‌شود تا هم‌ترازی محوری + فیوژن تطبیقی انجام شود.

'''
'''
4) مثال عددی ساده (2D برای دید شهودی)

فرض کنیم فقط یک باتچ و 2 کانال و تصویر 2×2 داریم (برای سادگی D=1 را حذف می‌کنیم):
x (اسکیپِ انکدر): شکل (Cx=2, H=2, W=2)
        x[0] = [[0.2, 0.4],
                [0.7, 0.1]]

        x[1] = [[0.5, 0.3],
                [0.9, 0.2]]
بعد از نگاشت‌های 1×1×1 و جمع و نرمال‌سازی و … فرض کن α به دست آمده:
        α (1-channel) = [[0.60, 0.20],
                         [0.90, 0.40]]
        (در عمل α از sigmoid می‌آید، پس بین 0 و 1 است.)

گیت‌کردن (ضرب broadcast): α روی هر دو کانال اعمال می‌شود:

        کانال 0:  x_gated[0] = x[0] ⊙ α
                   = [[0.2×0.60, 0.4×0.20],
                      [0.7×0.90, 0.1×0.40]]
                   = [[0.12, 0.08],
                      [0.63, 0.04]]
        کانال 1:  x_gated[1] = x[1] ⊙ α
                   = [[0.5×0.60, 0.3×0.20],
                      [0.9×0.90, 0.2×0.40]]
                   = [[0.30, 0.06],
                      [0.81, 0.08]]
شکل خروجی تغییر نکرده: همچنان (2, 2, 2). فقط مقادیر در هر وکسل بر اساس α کم‌وزیاد شده‌اند.

بعد از SkipGate چه می‌شود؟
        x_gated معمولاً وارد SkipProj(1×1×1) می‌شود تا Cx → C_dec هم‌کانال شود.
        سپس با فیچر دیکودر g وارد TCFCBlock می‌شود تا هم‌ترازی محوری + فیوژن تطبیقی انجام شود
'''

'\n4) مثال عددی ساده (2D برای دید شهودی)\n\nفرض کنیم فقط یک باتچ و 2 کانال و تصویر 2×2 داریم (برای سادگی D=1 را حذف می\u200cکنیم):\nx (اسکیپِ انکدر): شکل (Cx=2, H=2, W=2)\n        x[0] = [[0.2, 0.4],\n                [0.7, 0.1]]\n\n        x[1] = [[0.5, 0.3],\n                [0.9, 0.2]]\nبعد از نگاشت\u200cهای 1×1×1 و جمع و نرمال\u200cسازی و … فرض کن α به دست آمده:\n        α (1-channel) = [[0.60, 0.20],\n                         [0.90, 0.40]]\n        (در عمل α از sigmoid می\u200cآید، پس بین 0 و 1 است.)\n\nگیت\u200cکردن (ضرب broadcast): α روی هر دو کانال اعمال می\u200cشود:\n\n        کانال 0:  x_gated[0] = x[0] ⊙ α\n                   = [[0.2×0.60, 0.4×0.20],\n                      [0.7×0.90, 0.1×0.40]]\n                   = [[0.12, 0.08],\n                      [0.63, 0.04]]\n        کانال 1:  x_gated[1] = x[1] ⊙ α\n                   = [[0.5×0.60, 0.3×0.20],\n                      [0.9×0.90, 0.2×0.40]]\n                   = [[0.30, 0.06],\n                      [0.81, 0.08]]\nشکل خر

In [ ]:
class SkipProj(nn.Module):
    """
    پروجکشن اسکیپ‌ها به کانال هدف (مثلاً 64 برای @1/8 و 32 برای @1/4).
    - ورودی می‌تواند:
        الف) یک لیست/تاپل از 4 تنسورِ مدالیتی در همان سطح باشد: [t1, t1gd, t2, fl]، هرکدام (B, C_l, D,H,W)
        ب) یا یک تنسور از قبل concat شده باشد: (B, 4*C_l, D,H,W)
    - خروجی: (B, out_ch, D,H,W)
    """

    def __init__(self, in_ch: int, out_ch: int,
                 norm_type: str = "in", negative_slope: float = 0.1):
        super().__init__()
        def _norm(ch):
            if norm_type == "bn":   return nn.BatchNorm3d(ch)
            elif norm_type == "gn": return nn.GroupNorm(num_groups=8, num_channels=ch)
            else:                   return nn.InstanceNorm3d(ch, affine=True)  # "in"

        self.proj = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False),
            _norm(out_ch),
            nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
        )

        # init مناسب
        for m in self.modules():
            if isinstance(m, nn.Conv3d) and m.kernel_size == (1,1,1):
                nn.init.kaiming_uniform_(m.weight, a=1e-2)

    def forward(self, x):
        if isinstance(x, (list, tuple)):
            assert len(x) == 4, "SkipProj expects 4 tensors (t1,t1gd,t2,fl) at same level."
            # چک هم‌ابعادی (اختیاری)
            B,C,D,H,W = x[0].shape
            for t in x[1:]:
                assert t.shape[0] == B and t.shape[2:] == (D,H,W), "All skip tensors must align in (B,D,H,W)."
            x = torch.cat(x, dim=1)  # (B, 4*C_l, D,H,W)
        return self.proj(x)           # (B, out_ch, D,H,W)


In [ ]:
#@title عنوانِ این سلول
#@markdown (اختیاری) توضیح کوتاه اینجاست

'''
بعد از SkipGate، اسکیپ انکدر رو تمیز کرده‌ای: x_gated با شکل (B, Cx, D, H, W).
اما معمولاً کانال‌های اسکیپ (Cx) با کانال‌های جریان دیکودر در همان سطح (C_dec) یکی نیست.
SkipProj یک نگاشت ۱×۱×۱ می‌زند تا فقط محور کانال را از Cx → C_dec تبدیل کند؛
همهٔ ابعاد فضایی (D,H,W) دست‌نخورده می‌مانند. پس دقیقاً مثل یک Linear برای هر وکسل عمل می‌کند.

nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False)

        کانولوشن ۱×۱×۱ یعنی در هر وکسل (هر مختصات D,H,W) یک نگاشت خطی کانالی انجام بده.

        هیچ همسایه‌ای قاطی نمی‌شود؛ فقط محور C تغییر می‌کند (دقیقاً مثل یک Linear(in_ch→out_ch) برای هر وکسل).

        bias=False چون بلافاصله نرمال‌سازی داریم؛ بایاس نیازی نیست.

nn.InstanceNorm3d(out_ch, affine=True)

        برای 3D و batch کوچک، InstanceNorm پایدارتر از BatchNorm است.

        affine=True اجازه می‌دهد مقیاس/بایاس قابل‌آموزش باشد و افت اطلاعات نیفتد.

nn.LeakyReLU(negative_slope=0.1, inplace=True)

        مقداری نخطیّت اضافه می‌کند تا پس از نگاشت خطی، بیانگری‌مان افت نکند.

        LeakyReLU در 3D معمولاً از ReLU خشک‌تر پایدارتر است (گرادیان صفر گیر نمی‌کند).

این ماژول کجا می‌نشیند؟
        Upsample → g
        Skip (از انکدر) → SkipGate(x,g) → x_gated
        x_proj = SkipProj(x_gated)        ← کانال Cx → C_dec
        fused  = TCFCBlock(g, x_proj)     ← هم‌ترازی محوری + فیوژن
        out    = DecoderBlock(fused)

چرا این طراحی «به‌صرفه و کافی» است؟

        ما فقط می‌خواهیم اسکیپ را هم‌کانال کنیم—نه اینکه با همسایه‌ها مخلوط کنیم یا رزولوشن را دست بزنیم.
        ۱×۱×۱ در 3D ارزان‌ترین و مستقیم‌ترین راه است و بعدش IN+LeakyReLU کمک می‌کند سیگنال‌ها سالم و قابل‌آموزش بمانند.
        اگر همین‌جا 3×3×3 بزنی، پارامتر و حافظه بی‌جهت زیاد می‌شود و کار TCFC/DecoderBlock را تکراری می‌کنی.

اگر از اول Cx = C_dec بود، می‌توانی SkipProj را حذف کنی (یا فقط IN+Act نگه داری).'''

'\nبعد از SkipGate، اسکیپ انکدر رو تمیز کرده\u200cای: x_gated با شکل (B, Cx, D, H, W).\nاما معمولاً کانال\u200cهای اسکیپ (Cx) با کانال\u200cهای جریان دیکودر در همان سطح (C_dec) یکی نیست.\nSkipProj یک نگاشت ۱×۱×۱ می\u200cزند تا فقط محور کانال را از Cx → C_dec تبدیل کند؛\nهمهٔ ابعاد فضایی (D,H,W) دست\u200cنخورده می\u200cمانند. پس دقیقاً مثل یک Linear برای هر وکسل عمل می\u200cکند.\n\nnn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False)\n\n        کانولوشن ۱×۱×۱ یعنی در هر وکسل (هر مختصات D,H,W) یک نگاشت خطی کانالی انجام بده.\n\n        هیچ همسایه\u200cای قاطی نمی\u200cشود؛ فقط محور C تغییر می\u200cکند (دقیقاً مثل یک Linear(in_ch→out_ch) برای هر وکسل).\n\n        bias=False چون بلافاصله نرمال\u200cسازی داریم؛ بایاس نیازی نیست.\n\nnn.InstanceNorm3d(out_ch, affine=True)\n\n        برای 3D و batch کوچک، InstanceNorm پایدارتر از BatchNorm است.\n\n        affine=True اجازه می\u200cدهد مقیاس/بایاس قابل\u200cآموزش باشد و افت اطلاعات نیفتد.\n\nnn.LeakyReLU(negative_slope=0.1, inplace=True)\n\n     

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def _norm3d(ch: int, norm_type: str = "in") -> nn.Module:
    if norm_type == "bn": return nn.BatchNorm3d(ch)
    if norm_type == "gn": return nn.GroupNorm(8, ch)
    return nn.InstanceNorm3d(ch, affine=True)

class _AxisConv3D(nn.Module):
    """
    کانولوشن محوری 3D:
      mode="plain": Conv3d(out_ch, out_ch, k=(1,1,3)/(1,3,1)/(3,1,1))
      mode="dw":    Depthwise (groups=ch) + Pointwise(1×1×1)
    """
    def __init__(self, ch: int, kernel: tuple[int,int,int], depthwise: bool, norm_type: str, neg_slope: float):
        super().__init__()
        if depthwise:
            self.block = nn.Sequential(
                nn.Conv3d(ch, ch, kernel_size=kernel, padding=tuple(k//2 for k in kernel),
                          groups=ch, bias=False),
                _norm3d(ch, norm_type),
                nn.LeakyReLU(neg_slope, inplace=True),
                nn.Conv3d(ch, ch, kernel_size=1, bias=False),
                _norm3d(ch, norm_type),
                nn.LeakyReLU(neg_slope, inplace=True),
            )
        else:
            self.block = nn.Sequential(
                nn.Conv3d(ch, ch, kernel_size=kernel, padding=tuple(k//2 for k in kernel), bias=False),
                _norm3d(ch, norm_type),
                nn.LeakyReLU(neg_slope, inplace=True),
            )
    def forward(self, x): return self.block(x)

class TCFCBlock(nn.Module):
    """
    Translation-Compensated Feature Calibration
    main: فیچرِ مسیر دیکودر (بعد از Upsample)
    trans: اسکیپِ ترنس/انکدر (پس از پروجکشن)
    خروجی: فیچر تطبیق‌یافته با کانال out_ch و همان رزولوشن
    """
    def __init__(self,
                 main_ch: int, trans_ch: int, out_ch: int,
                 depthwise: bool = False, norm_type: str = "in",
                 neg_slope: float = 0.1, dropout: float = 0.0):
        super().__init__()
        self.neg_slope = neg_slope

        # هر دو شاخه → فضای مشترک out_ch
        self.main_proj  = nn.Sequential(nn.Conv3d(main_ch,  out_ch, 1, bias=False),
                                        _norm3d(out_ch, norm_type),
                                        nn.LeakyReLU(neg_slope, inplace=True))
        self.trans_proj = nn.Sequential(nn.Conv3d(trans_ch, out_ch, 1, bias=False),
                                        _norm3d(out_ch, norm_type),
                                        nn.LeakyReLU(neg_slope, inplace=True))

        # فیلترهای محوری روی هر کدام (x,y,z)
        self.m_x = _AxisConv3D(out_ch, (1,1,3), depthwise, norm_type, neg_slope)
        self.m_y = _AxisConv3D(out_ch, (1,3,1), depthwise, norm_type, neg_slope)
        self.m_z = _AxisConv3D(out_ch, (3,1,1), depthwise, norm_type, neg_slope)

        self.t_x = _AxisConv3D(out_ch, (1,1,3), depthwise, norm_type, neg_slope)
        self.t_y = _AxisConv3D(out_ch, (1,3,1), depthwise, norm_type, neg_slope)
        self.t_z = _AxisConv3D(out_ch, (3,1,1), depthwise, norm_type, neg_slope)

        # فیوژن و ساخت گیت  (۶×out_ch → out_ch → Sigmoid)
        self.fuse_gate = nn.Sequential(
            nn.Conv3d(6*out_ch, out_ch, 1, bias=False),
            _norm3d(out_ch, norm_type),
            nn.LeakyReLU(neg_slope, inplace=True),
            nn.Conv3d(out_ch, out_ch, 1, bias=True),
            nn.Sigmoid()
        )

        self.dropout = nn.Dropout3d(dropout) if dropout>0 else nn.Identity()

    def forward(self, main: torch.Tensor, trans: torch.Tensor) -> torch.Tensor:
        # هم‌ترازی فضایی
        if main.shape[-3:] != trans.shape[-3:]:
            main = F.interpolate(main, size=trans.shape[-3:], mode="trilinear", align_corners=False)

        m = self.main_proj(main)   # (B,out_ch, D,H,W)
        t = self.trans_proj(trans) # (B,out_ch, D,H,W)

        # مسیرهای محوری
        mx, my, mz = self.m_x(m), self.m_y(m), self.m_z(m)
        tx, ty, tz = self.t_x(t), self.t_y(t), self.t_z(t)

        fusion = torch.cat([mx, my, mz, tx, ty, tz], dim=1)  # (B, 6*out_ch, ...)
        gate   = self.fuse_gate(fusion)                      # (B, out_ch, ..., ) ∈ (0..1)

        # کالیبراسیون تطبیقی: «trans» را فیلتر کن و به «main» بیفزا
        out = m + gate * t
        out = self.dropout(out)
        return out


In [ ]:
#@title تست
#@markdown

'''
B, C, D, H, W = 1, 64, 28, 28, 20
main = torch.randn(B, C, D, H, W)
skip = torch.randn(B, 48, D, H, W)   # فرض کن کانال اسکیپ فرق دارد

# 1) گیت و پروجکشن اسکیپ
gate = SkipGate(in_ch_x=48, in_ch_g=64, inter_ch=16)
proj = SkipProj(in_ch=48, out_ch=64)

skip_g = gate(skip, main)    # (B,48,D,H,W)
skip_p = proj(skip_g)        # (B,64,D,H,W)

# 2) TCFC
tcfc = TCFCBlock(channels=64, depthwise=False)
out = tcfc(main, skip_p)     # (B,64,D,H,W)
print(out.shape)             # torch.Size([1, 64, 28, 28, 20])
'''

'\nB, C, D, H, W = 1, 64, 28, 28, 20\nmain = torch.randn(B, C, D, H, W)\nskip = torch.randn(B, 48, D, H, W)   # فرض کن کانال اسکیپ فرق دارد\n\n# 1) گیت و پروجکشن اسکیپ\ngate = SkipGate(in_ch_x=48, in_ch_g=64, inter_ch=16)\nproj = SkipProj(in_ch=48, out_ch=64)\n\nskip_g = gate(skip, main)    # (B,48,D,H,W)\nskip_p = proj(skip_g)        # (B,64,D,H,W)\n\n# 2) TCFC\ntcfc = TCFCBlock(channels=64, depthwise=False)\nout = tcfc(main, skip_p)     # (B,64,D,H,W)\nprint(out.shape)             # torch.Size([1, 64, 28, 28, 20])\n'

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int,
                 norm_type: str = "in", neg_slope: float = 0.1,
                 dropout: float = 0.0, use_residual: bool = True):
        super().__init__()
        self.use_residual = use_residual
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            _norm3d(out_ch, norm_type),
            nn.LeakyReLU(neg_slope, inplace=True),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            _norm3d(out_ch, norm_type),
            nn.LeakyReLU(neg_slope, inplace=True),
            nn.Dropout3d(dropout) if dropout>0 else nn.Identity(),
        )
        self.proj = nn.Conv3d(in_ch, out_ch, 1, bias=False) if (use_residual and in_ch!=out_ch) else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x if isinstance(self.proj, nn.Identity) else self.proj(x)
        y = self.block(x)
        return y + identity if self.use_residual else y


In [ ]:
#@title تست
#@markdown

'''
# ورودی @1/4 مثالی: (B,C,D,H,W) = (1,64,40,56,56)
x = torch.randn(1, 64, 40, 56, 56)

blk_same = DecoderBlock(64, 64, use_residual=True, dropout=0.0)
y1 = blk_same(x)
print(y1.shape)  # torch.Size([1, 64, 40, 56, 56])

blk_change = DecoderBlock(64, 32, use_residual=True, dropout=0.1)
y2 = blk_change(x)
print(y2.shape)  # torch.Size([1, 32, 40, 56, 56])'''


'\n# ورودی @1/4 مثالی: (B,C,D,H,W) = (1,64,40,56,56)\nx = torch.randn(1, 64, 40, 56, 56)\n\nblk_same = DecoderBlock(64, 64, use_residual=True, dropout=0.0)\ny1 = blk_same(x)\nprint(y1.shape)  # torch.Size([1, 64, 40, 56, 56])\n\nblk_change = DecoderBlock(64, 32, use_residual=True, dropout=0.1)\ny2 = blk_change(x)\nprint(y2.shape)  # torch.Size([1, 32, 40, 56, 56])'

In [ ]:
class DecoderStage(nn.Module):
    """
    یک استیج دیکودر:
      x_in --Upsample--> x_up --SkipGate(skip)--> s_gated --TCFC(x_up, s_gated)--> fused --DecoderBlock--> x_out
    خروجی: x_out  (و معمولاً برای deep supervision از x_outِ برخی استیج‌ها Logits کمکی می‌گیریم)
    """
    def __init__(self,
                 x_in_ch: int,          # کانال ورودی استیج (از استیج قبلی/باتلنک)
                 x_up_ch: int,          # کانال بعد از up/proj (ورودی TCFC)
                 skip_ch: int,          # کانال اسکیپ (بعد از SkipProj)
                 out_ch: int,           # کانال خروجی استیج
                 up_method: str = "interp",
                 norm_type: str = "in",
                 neg_slope: float = 0.1,
                 depthwise_tcfc: bool = False,
                 dropout_tcfc: float = 0.0,
                 dropout_dec: float = 0.0):
        super().__init__()
        # Upsample به کانال x_up_ch
        self.up = UpsampleBlock(in_ch=x_in_ch, out_ch=x_up_ch,
                                method=up_method, scale=(2,2,2),
                                norm_type=norm_type, negative_slope=neg_slope)
        # گیت اسکیپ: کانال دیکودر=x_up_ch، کانال اسکیپ=skip_ch
        self.skip_gate = SkipGate(x_ch=x_up_ch, s_ch=skip_ch,
                                  inter_ch=None, use_channel_gate=True,
                                  norm_type=norm_type, negative_slope=neg_slope)
        # TCFC: x_up_ch (main), skip_ch (trans) → out_ch_tcfc (برای سادگی = x_up_ch)
        self.tcfc = TCFCBlock(main_ch=x_up_ch, trans_ch=skip_ch, out_ch=x_up_ch,
                              depthwise=depthwise_tcfc, norm_type=norm_type,
                              neg_slope=neg_slope, dropout=dropout_tcfc)
        # DecoderBlock: x_up_ch → out_ch
        self.dec = DecoderBlock(in_ch=x_up_ch, out_ch=out_ch,
                                norm_type=norm_type, neg_slope=neg_slope,
                                dropout=dropout_dec, use_residual=True)

    def forward(self, x_in: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        # 1) upsample تا سایزِ skip
        x_up = self.up(x_in, target_size=skip.shape[-3:])
        # 2) گیت اسکیپ با توجه به x_up
        s_gated = self.skip_gate(x_up, skip)
        # 3) تطبیق و فیوژن تطبیقی
        fused = self.tcfc(x_up, s_gated)
        # 4) پالایش خروجی استیج
        out = self.dec(fused)
        return out


In [ ]:
#@title تست
#@markdown
'''
# فرض ورودی Bottleneck @1/16:
x16 = torch.randn(1, 128, 10, 14, 14)
# اسکیپ‌های انکدر:
skip_1_8 = torch.randn(1, 64, 20, 28, 28)
skip_1_4 = torch.randn(1, 32, 40, 56, 56)

# Stage-1: 1/16 → 1/8  (با اسکیپ@1/8)
s1 = DecoderStage(in_ch=128, out_ch=64, has_skip=True, skip_ch=64)
y_1_8 = s1(x16, skip_1_8)                # → (1, 64, 20, 28, 28)
print("Stage-1:", y_1_8.shape)

# Stage-2: 1/8 → 1/4   (با اسکیپ@1/4)
s2 = DecoderStage(in_ch=64, out_ch=32, has_skip=True, skip_ch=32)
y_1_4 = s2(y_1_8, skip_1_4)              # → (1, 32, 40, 56, 56)
print("Stage-2:", y_1_4.shape)

# Stage-3: 1/4 → 1/2   (بدون اسکیپ)
s3 = DecoderStage(in_ch=32, out_ch=16, has_skip=False)
y_1_2 = s3(y_1_4)                         # → (1, 16, 80, 112, 112)
print("Stage-3:", y_1_2.shape)
'''


'\n# فرض ورودی Bottleneck @1/16:\nx16 = torch.randn(1, 128, 10, 14, 14)\n# اسکیپ\u200cهای انکدر:\nskip_1_8 = torch.randn(1, 64, 20, 28, 28)\nskip_1_4 = torch.randn(1, 32, 40, 56, 56)\n\n# Stage-1: 1/16 → 1/8  (با اسکیپ@1/8)\ns1 = DecoderStage(in_ch=128, out_ch=64, has_skip=True, skip_ch=64)\ny_1_8 = s1(x16, skip_1_8)                # → (1, 64, 20, 28, 28)\nprint("Stage-1:", y_1_8.shape)\n\n# Stage-2: 1/8 → 1/4   (با اسکیپ@1/4)\ns2 = DecoderStage(in_ch=64, out_ch=32, has_skip=True, skip_ch=32)\ny_1_4 = s2(y_1_8, skip_1_4)              # → (1, 32, 40, 56, 56)\nprint("Stage-2:", y_1_4.shape)\n\n# Stage-3: 1/4 → 1/2   (بدون اسکیپ)\ns3 = DecoderStage(in_ch=32, out_ch=16, has_skip=False)\ny_1_2 = s3(y_1_4)                         # → (1, 16, 80, 112, 112)\nprint("Stage-3:", y_1_2.shape)\n'

In [ ]:
import torch
import torch.nn as nn

class Decoder(nn.Module):
    """
    Stage-1: 1/16 → 1/8   (با اسکیپ @1/8)
    Stage-2: 1/8  → 1/4   (با اسکیپ @1/4)
    Stage-3: 1/4  → 1/2   (بدون اسکیپ)
    Stage-4: 1/2  → 1×    (بدون اسکیپ)

    خروجی‌ها:
      y_1x  : فول‌رز
      y_1_8 : رز @1/8 (برای aux)
      y_1_4 : رز @1/4 (برای aux)
    """
    def __init__(
        self,
        *,  # فقط keyword
        ch_s1_in: int,   # کانال ورودی bottleneck (L4 واقعی)
        ch_s1_out: int,  # کانال خروجی Stage-1 (و ورودی Stage-2)
        skip_s1_ch: int, # کانال اسکیپِ پروجکت‌شده @1/8
        ch_s2_out: int,  # کانال خروجی Stage-2 (و ورودی Stage-3)
        skip_s2_ch: int, # کانال اسکیپِ پروجکت‌شده @1/4
        ch_s3_out: int,  # کانال خروجی Stage-3
        ch_s4_out: int,  # کانال خروجی Stage-4 (ورودی SegHead)
        up_method: str = "interp",
        negative_slope: float = 0.1,
        tcfc_depthwise: bool = False,
        dec_dropout: float = 0.0,
        norm_type: str = "in",
    ):
        super().__init__()

        # Stage-1: با اسکیپ@1/8
        self.s1 = DecoderStage(
            x_in_ch=ch_s1_in, x_up_ch=ch_s1_out, skip_ch=skip_s1_ch, out_ch=ch_s1_out,
            up_method=up_method, norm_type=norm_type, neg_slope=negative_slope,
            depthwise_tcfc=tcfc_depthwise, dropout_tcfc=dec_dropout, dropout_dec=dec_dropout
        )
        # Stage-2: با اسکیپ@1/4
        self.s2 = DecoderStage(
            x_in_ch=ch_s1_out, x_up_ch=ch_s2_out, skip_ch=skip_s2_ch, out_ch=ch_s2_out,
            up_method=up_method, norm_type=norm_type, neg_slope=negative_slope,
            depthwise_tcfc=tcfc_depthwise, dropout_tcfc=dec_dropout, dropout_dec=dec_dropout
        )
        # Stage-3: بدون اسکیپ (Upsample + DecoderBlock)
        self.up3  = UpsampleBlock(in_ch=ch_s2_out, out_ch=ch_s3_out,
                                  method=up_method, scale=(2,2,2),
                                  norm_type=norm_type, negative_slope=negative_slope,
                                  dropout=dec_dropout)
        self.dec3 = DecoderBlock(in_ch=ch_s3_out, out_ch=ch_s3_out,
                                 norm_type=norm_type, neg_slope=negative_slope,
                                 dropout=dec_dropout, use_residual=True)

        # Stage-4: بدون اسکیپ
        self.up4  = UpsampleBlock(in_ch=ch_s3_out, out_ch=ch_s4_out,
                                  method=up_method, scale=(2,2,2),
                                  norm_type=norm_type, negative_slope=negative_slope,
                                  dropout=dec_dropout)
        self.dec4 = DecoderBlock(in_ch=ch_s4_out, out_ch=ch_s4_out,
                                 norm_type=norm_type, neg_slope=negative_slope,
                                 dropout=dec_dropout, use_residual=True)

    def forward(self, x_1_16, skip_1_8, skip_1_4):
        # Stage-1: 1/16→1/8 (با هم‌ترازی دقیق با skip_1_8 در داخل Stage)
        y_1_8 = self.s1(x_1_16, skip_1_8)
        # Stage-2: 1/8→1/4
        y_1_4 = self.s2(y_1_8,  skip_1_4)
        # Stage-3: 1/4→1/2 (بدون اسکیپ)
        y_1_2 = self.dec3(self.up3(y_1_4))
        # Stage-4: 1/2→1× (بدون اسکیپ)
        y_1x  = self.dec4(self.up4(y_1_2))
        return y_1x, y_1_8, y_1_4


In [ ]:
#@title تست
#@markdown

'''
import torch

def test_decoder_smoke(
    ch_s1_in=32,   # ورودی دیکودر @1/16 (مثلاً همون base_ch)
    ch_s1_out=64,  # خروجی Stage-1 و کانال هدف skip@1/8
    ch_s2_out=32,  # خروجی Stage-2 و کانال هدف skip@1/4
    ch_s3_out=16,  # خروجی Stage-3
    ch_s4_out=16,  # خروجی Stage-4 (ورودی SegHead)
    up_method="interp",
    tcfc_depthwise=False,
    dec_dropout=0.0,
    device="cpu"
):
    # ابعاد فضایی سازگار با ورودی 224×224×160:
    # 1/16 ≈ 14×14×10 , 1/8 ≈ 28×28×20 , 1/4 ≈ 56×56×40
    D16, H16, W16 = 10, 14, 14   # bottleneck (1/16)
    D8,  H8,  W8  = 20, 28, 28   # skip@1/8
    D4,  H4,  W4  = 40, 56, 56   # skip@1/4

    # 1) ساخت ورودی‌ها
    x_1_16   = torch.randn(1, ch_s1_in, D16, H16, W16, device=device)
    skip_1_8 = torch.randn(1, ch_s1_out, D8,  H8,  W8,  device=device)  # باید با ch_s1_out هم‌کانال باشد
    skip_1_4 = torch.randn(1, ch_s2_out, D4,  H4,  W4,  device=device)  # باید با ch_s2_out هم‌کانال باشد

    # 2) ساخت دیکودر
    dec = Decoder(
        ch_s1_in=ch_s1_in, ch_s1_out=ch_s1_out, skip_s1_ch=ch_s1_out,
        ch_s2_out=ch_s2_out, skip_s2_ch=ch_s2_out,
        ch_s3_out=ch_s3_out, ch_s4_out=ch_s4_out,
        up_method=up_method, tcfc_depthwise=tcfc_depthwise, dec_dropout=dec_dropout
    ).to(device).eval()

    # 3) فوروارد و چک ابعاد
    with torch.no_grad():
        y_1x, y_1_8, y_1_4 = dec(x_1_16, skip_1_8, skip_1_4)

    print("y_1_8:", tuple(y_1_8.shape))  # انتظار: (1, ch_s1_out,  D8, H8, W8)
    print("y_1_4:", tuple(y_1_4.shape))  # انتظار: (1, ch_s2_out,  D4, H4, W4)
    print("y_1x :", tuple(y_1x.shape))   # انتظار: (1, ch_s4_out, 160,224,224) اگر 4 استیج داشتی تا فول‌رز
                                         # در این تست ساده 4 استیج تا 1× داریم: (D=80→160، H=112→224، W=112→224)

    # چون ما 4 استیج داریم (1/16→1/8→1/4→1/2→1×)، انتظار نهایی:
    D1, H1, W1 = D16*16//16, H16*16//16, W16*16//16  # فقط برای اطلاع، با مسیر آپ‌سمپل ما باید 160,224,224 شود
    assert y_1_8.shape[1] == ch_s1_out and y_1_8.shape[2:] == (D8, H8, W8),   "Stage-1 خروجی/ابعاد درست نیست"
    assert y_1_4.shape[1] == ch_s2_out and y_1_4.shape[2:] == (D4, H4, W4),   "Stage-2 خروجی/ابعاد درست نیست"
    assert y_1x.shape[1]  == ch_s4_out and y_1x.shape[2:]  == (160, 224, 224), "Stage-4 باید فول‌رز باشد"

    print("✅ Decoder smoke test passed.")

# اجرا:
test_decoder_smoke()
'''

'\nimport torch\n\ndef test_decoder_smoke(\n    ch_s1_in=32,   # ورودی دیکودر @1/16 (مثلاً همون base_ch)\n    ch_s1_out=64,  # خروجی Stage-1 و کانال هدف skip@1/8\n    ch_s2_out=32,  # خروجی Stage-2 و کانال هدف skip@1/4\n    ch_s3_out=16,  # خروجی Stage-3\n    ch_s4_out=16,  # خروجی Stage-4 (ورودی SegHead)\n    up_method="interp",\n    tcfc_depthwise=False,\n    dec_dropout=0.0,\n    device="cpu"\n):\n    # ابعاد فضایی سازگار با ورودی 224×224×160:\n    # 1/16 ≈ 14×14×10 , 1/8 ≈ 28×28×20 , 1/4 ≈ 56×56×40\n    D16, H16, W16 = 10, 14, 14   # bottleneck (1/16)\n    D8,  H8,  W8  = 20, 28, 28   # skip@1/8\n    D4,  H4,  W4  = 40, 56, 56   # skip@1/4\n\n    # 1) ساخت ورودی\u200cها\n    x_1_16   = torch.randn(1, ch_s1_in, D16, H16, W16, device=device)\n    skip_1_8 = torch.randn(1, ch_s1_out, D8,  H8,  W8,  device=device)  # باید با ch_s1_out هم\u200cکانال باشد\n    skip_1_4 = torch.randn(1, ch_s2_out, D4,  H4,  W4,  device=device)  # باید با ch_s2_out هم\u200cکانال باشد\n\n    # 2) ساخت دیکود

In [ ]:
# ❇️ SegmentationModel

In [ ]:
class SegHead(nn.Module):
    """
    سر نهایی سگمنتیشن:
      - Conv1×1×1 برای تولید لاجیت کلاس‌ها
      - اگر upsample_to_full=True باشد، در forward اندازهٔ مقصد را می‌گیرد.
    """
    def __init__(self, in_ch: int, num_classes: int, upsample_to_full: bool = False):
        super().__init__()
        self.upsample_to_full = upsample_to_full
        self.conv = nn.Conv3d(in_ch, num_classes, kernel_size=1, bias=True)

    def forward(self, x, full_size=None):
        logits = self.conv(x)
        if self.upsample_to_full:
            assert full_size is not None, "full_size باید برای upsample_to_full=True داده شود"
            # full_size ترتیب (D,H,W)
            logits = nn.functional.interpolate(logits, size=full_size,
                                               mode="trilinear", align_corners=False)
        return logits


In [ ]:
#@title تست
#@markdown

'''
x = torch.randn(1, 32, 224, 224, 160)   # خروجی Stage-4 @ فول‌رز
head = SegHead(in_ch=32, num_classes=4, upsample_to_full=False)
logits = head(x)
print(logits.shape)  # → (1, 4, 224, 224, 160)
'''

'\nx = torch.randn(1, 32, 224, 224, 160)   # خروجی Stage-4 @ فول\u200cرز\nhead = SegHead(in_ch=32, num_classes=4, upsample_to_full=False)\nlogits = head(x)\nprint(logits.shape)  # → (1, 4, 224, 224, 160)\n'

In [ ]:
'''
ایده: از خروجی استیج‌های میانی (مثلاً @1/8 و @1/4) هم logits بگیر و به فول‌رز upsample کن تا در loss کمک کنند (همگرایی بهتر، گرادیان‌های عمیق‌تر).
'''

'\nایده: از خروجی استیج\u200cهای میانی (مثلاً @1/8 و @1/4) هم logits بگیر و به فول\u200cرز upsample کن تا در loss کمک کنند (همگرایی بهتر، گرادیان\u200cهای عمیق\u200cتر).\n'

In [ ]:
class AuxHead(nn.Module):
    """
    لاجیت کمکی برای deep supervision:
      - Conv1×1×1 → interpolate به فول‌رز جهت محاسبهٔ loss کمکی
    """
    def __init__(self, in_ch: int, num_classes: int):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, num_classes, kernel_size=1, bias=True)

    def forward(self, x, full_size):
        logits = self.conv(x)
        logits = nn.functional.interpolate(
            logits, size=full_size, mode="trilinear", align_corners=False
        )
        return logits


In [ ]:
#@title تست
#@markdown

'''
# فرض: خروجی Stage-2 @1/4
x_mid = torch.randn(1, 32, 40, 56, 56)
aux = AuxHead(in_ch=32, num_classes=4)
logits_aux = aux(x_mid, full_size=(160, 224, 224))
print(logits_aux.shape)  # → (1, 4, 160, 224, 224)
'''

'\n# فرض: خروجی Stage-2 @1/4\nx_mid = torch.randn(1, 32, 40, 56, 56)\naux = AuxHead(in_ch=32, num_classes=4)\nlogits_aux = aux(x_mid, full_size=(160, 224, 224))\nprint(logits_aux.shape)  # → (1, 4, 160, 224, 224)\n'

In [ ]:
# 💠 Multi Segment Model

In [ ]:
def init_weights_3d(m, leaky_a: float = 0.1):
    import torch.nn as nn
    if isinstance(m, nn.Conv3d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="leaky_relu", a=leaky_a)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.ConvTranspose3d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="leaky_relu", a=leaky_a)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        # برای Q/K/V/Proj در Attention که بدون nonlinearity مستقیم عمل می‌کنند
        nn.init.xavier_uniform_(m.weight)  # gain=1
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, (nn.BatchNorm3d, nn.InstanceNorm3d, nn.GroupNorm, nn.LayerNorm)):
        if hasattr(m, "weight") and m.weight is not None:
            nn.init.ones_(m.weight)
        if hasattr(m, "bias") and m.bias is not None:
            nn.init.zeros_(m.bias)
    # اگر ماژولی پارامتر rel_bias داشته باشد، صفر نگهش دار:
    if hasattr(m, "rel_bias"):
        with torch.no_grad():
            m.rel_bias.zero_()


In [ ]:
class MultiModalSegNet(nn.Module):
    """
    ورودی:  (B, 4, H, W, D) ← [T1, T1Gd, T2, FLAIR]
    خروجی:  {"logits": (B, num_classes, H, W, D),  +  aux1/aux2 در حالت train}
    """
    def __init__(
        self,
        base_ch: int = 32,
        num_heads: int = 4,
        num_classes: int = 4,
        up_method: str = "interp",
        use_gating: bool = True,
        bottleneck_heads: int = 4,
        max_len: int = 512,
        negative_slope: float = 0.1,
        deep_supervision: bool = True,
        tcfc_depthwise: bool = True,
        # افزوده‌ها:
        dropout: float = 0.0,                 # ← «پارامتر سراسری» که کاربر می‌دهد
        attn_dropout: float | None = None,    # اگر None بود از dropout استفاده می‌کنیم
        proj_dropout: float | None = None,
        ffn_dropout: float | None = None,
        dec_dropout: float | None = None,     # اگر None بود از dropout استفاده می‌کنیم
        norm_type: str = "in",
        ):
        super().__init__()
        # --- پارامترها را اول ذخیره کن
        self.num_classes     = num_classes
        self.deep_supervision= deep_supervision
        self.negative_slope  = negative_slope
        self.up_method       = up_method
        self.tcfc_depthwise  = tcfc_depthwise
        self.dec_dropout     = dec_dropout
        self.max_len         = max_len
        self.norm_type       = norm_type
        self.bottleneck_heads= bottleneck_heads   # ← مهم
        # wiring سراسری
        self.dropout       = float(dropout)
        self.attn_dropout  = self.dropout if attn_dropout is None else float(attn_dropout)
        self.proj_dropout  = self.dropout if proj_dropout is None else float(proj_dropout)
        self.ffn_dropout   = self.dropout if ffn_dropout  is None else float(ffn_dropout)
        self.dec_dropout   = self.dropout if dec_dropout  is None else float(dec_dropout)


        # --- 1) اول انکدر را بساز (تا AttributeError نگیری)
        self.encoder = DualBranchEncoder(
        in_channels=1, base_channels=base_ch,
        num_heads=num_heads, max_len=max_len, use_gating=use_gating,
        attn_dropout=self.attn_dropout, proj_dropout=self.proj_dropout,
        ffn_dropout=self.ffn_dropout,   # ← اضافه شود
        prenorm=False
        )


        '''self.encoder = DualBranchEncoder(
            in_channels=1, base_channels=base_ch,
            num_heads=num_heads, use_gating=use_gating
        )'''

        # --- 2) حالا بر اساس خروجی واقعی انکدر بقیه را بساز
        self._build_from_encoder_probe()

        self.apply(lambda m: init_weights_3d(m, leaky_a=self.negative_slope))

    def _build_from_encoder_probe(self):
        # پروب کانال‌ها و ابعاد L2/L3/L4 از انکدر
        dev = next(self.encoder.parameters()).device
        was_training = self.encoder.training
        self.encoder.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 16, 16, 16, device=dev)  # سایز کوچک کافی است
            enc = self.encoder(dummy, dummy, dummy, dummy)
            t1_L2, t1_L3, t1_L4 = enc["t1"]
            C2, C3, C4 = t1_L2.shape[1], t1_L3.shape[1], t1_L4.shape[1]
            D4, H4, W4 = t1_L4.shape[2], t1_L4.shape[3], t1_L4.shape[4]
        self.encoder.train(was_training)

        safe_max_len = max(self.max_len, D4, H4, W4)

        # 3) Bottleneck با کانال واقعی L4
        '''self.bottleneck = BottleneckBlock(
            dim=C4, num_heads=self.bottleneck_heads, max_len=safe_max_len,
            use_gating=True, use_self_refine=True,
            norm_type=self.norm_type, negative_slope=self.negative_slope
        )'''
        self.bottleneck = BottleneckBlock(
        dim=C4, num_heads=self.bottleneck_heads, max_len=safe_max_len,
        use_gating=True, use_self_refine=True,
        attn_dropout=self.attn_dropout, proj_dropout=self.proj_dropout,
        prenorm=False, use_rel_bias=True,
        norm_type=self.norm_type, negative_slope=self.negative_slope,
        ffn_dropout=self.ffn_dropout      # ← اضافه شود
        )



        # 4) SkipProj ها (concat 4 modality → 1×1×1 → کانال هدف دیکودر)
        self.skip1_proj = SkipProj(in_ch=4*C3, out_ch=64, norm_type=self.norm_type,
                                   negative_slope=self.negative_slope)  # @1/8
        self.skip2_proj = SkipProj(in_ch=4*C2, out_ch=32, norm_type=self.norm_type,
                                   negative_slope=self.negative_slope)  # @1/4

        # 5) Decoder سازگار با کانال‌ها
        self.decoder = Decoder(
            ch_s1_in=C4, ch_s1_out=64, skip_s1_ch=64,   # 1/16→1/8
            ch_s2_out=32, skip_s2_ch=32,               # 1/8→1/4
            ch_s3_out=16, ch_s4_out=16,                # 1/4→1/2→1×
            # 🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️🤷‍♀️

            up_method=self.up_method, negative_slope=self.negative_slope,
            tcfc_depthwise=self.tcfc_depthwise, dec_dropout=self.dec_dropout,
            norm_type=self.norm_type
        )

        # 6) سرها
        self.seg_head = SegHead(in_ch=16, num_classes=self.num_classes, upsample_to_full=False)
        if self.deep_supervision:
            self.aux1 = AuxHead(in_ch=64, num_classes=self.num_classes)  # از y_1_8
            self.aux2 = AuxHead(in_ch=32, num_classes=self.num_classes)  # از y_1_4

    def forward(self, x):
        """
        x: (B, 4, H, W, D) ← [T1, T1Gd, T2, FLAIR]
        """
        B, C4, H, W, D = x.shape
        assert C4 == 4, "ورودی باید ۴ مدالیتی باشد."
        t1, t1gd, t2, fl = x[:,0:1], x[:,1:2], x[:,2:3], x[:,3:4]

        #print('================ prep =====================')

        # Encoder
        enc = self.encoder(t1, t1gd, t2, fl)
        t1_L2, t1_L3, t1_L4       = enc['t1']
        t1gd_L2, t1gd_L3, t1gd_L4 = enc['t1gd']
        t2_L2, t2_L3, t2_L4       = enc['t2']
        fl_L2, fl_L3, fl_L4       = enc['flair']

        #print('================ Encoder =====================')

        # Bottleneck @1/16
        fused_1_16, _, _ = self.bottleneck(t1_L4, t1gd_L4, t2_L4, fl_L4)

        #print('================ Bottleneck =====================')

        # Skips
        skip_1_8 = self.skip1_proj([t1_L3, t1gd_L3, t2_L3, fl_L3])  # (B,64,1/8)
        skip_1_4 = self.skip2_proj([t1_L2, t1gd_L2, t2_L2, fl_L2])  # (B,32,1/4)

        #print('================ Skips =====================')

        # Decoder
        y_1x, y_1_8, y_1_4 = self.decoder(fused_1_16, skip_1_8, skip_1_4)

        #print('================ Decoder =====================')

        # Heads
        logits = self.seg_head(y_1x)

        #print('================ Heads =====================')

        if self.deep_supervision and self.training:
            aux1 = self.aux1(y_1_8, full_size=(D, H, W))
            aux2 = self.aux2(y_1_4, full_size=(D, H, W))
            return {"logits": logits, "aux1": aux1, "aux2": aux2}
            #print('================ deep_supervision =====================')
        return {"logits": logits}


In [ ]:

#@title skip this module
#@markdown


class MultiModalSegNet_prev(nn.Module):
    """
    ورودی: (B, 4, H, W, D)  ← [T1, T1Gd, T2, FLAIR]
    خروجی: logits فول‌رز (B, num_classes, H, W, D)
    """
    def __init__(
        self,
        base_ch: int = 32,           # خروجی ConvStem (در کد شما ثابت است در L2/L3/L4)
        num_heads: int = 4,
        num_classes: int = 4,
        up_method: str = "interp",
        use_gating: bool = True,
        bottleneck_heads: int = 4,
        max_len: int = 256,
        negative_slope: float = 0.1,
        deep_supervision: bool = True,
        tcfc_depthwise: bool = False,
        dec_dropout: float = 0.0,
    ):
        super().__init__()

        C = base_ch
        # 1) انکدر دو-شاخه (طبق کد شما: خروجی هر سطح = C ثابت)
        self.encoder = DualBranchEncoder(in_channels=1, base_channels=C, num_heads=num_heads, use_gating=use_gating)

        # 2) فیوژن bottleneck (1/16)
        self.bottleneck = BottleneckBlock(
            dim=C, num_heads=bottleneck_heads, max_len=max_len,
            use_gating=True, use_self_refine=True
        )

        # 3) فیوژن اسکیپ‌ها (concat 4 modality → 1×1×1 → کانال هدفِ دیکودر)
        #   - برای Stage-1 (1/8): هدف کانال = 64 (مثالی)
        #   - برای Stage-2 (1/4): هدف کانال = 32
        self.skip1_proj = nn.Sequential(
            nn.Conv3d(4*C, 64, kernel_size=1, bias=False),
            nn.InstanceNorm3d(64, affine=True),
            nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
        )
        self.skip2_proj = nn.Sequential(
            nn.Conv3d(4*C, 32, kernel_size=1, bias=False),
            nn.InstanceNorm3d(32, affine=True),
            nn.LeakyReLU(negative_slope=negative_slope, inplace=True),
        )

        # 4) دیکودر (با کانال‌بندی سازگار با بالا)
        self.decoder = Decoder(
            ch_s1_in=C,    ch_s1_out=64, skip_s1_ch=64,   # 1/16→1/8  (+skip@1/8 proj به 64)
            ch_s2_out=32,  skip_s2_ch=32,                 # 1/8→1/4   (+skip@1/4 proj به 32)
            ch_s3_out=16,                                  # 1/4→1/2
            ch_s4_out=16,                                  # 1/2→1×
            up_method=up_method, negative_slope=negative_slope,
            tcfc_depthwise=tcfc_depthwise, dec_dropout=dec_dropout
        )

        # 5) هدها
        self.seg_head = SegHead(in_ch=16, num_classes=num_classes, upsample_to_full=False)
        self.deep_supervision = deep_supervision
        if deep_supervision:
            self.aux1 = AuxHead(in_ch=64, num_classes=num_classes)  # from y_1_8
            self.aux2 = AuxHead(in_ch=32, num_classes=num_classes)  # from y_1_4

        # init ساده برای پروجکشن‌ها
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                if m.kernel_size == (1,1,1):
                    nn.init.kaiming_uniform_(m.weight, a=1e-2)

    def forward(self, x):
        """
        x: (B, 4, H, W, D)  ← ترتیب کانالی: [T1, T1Gd, T2, FLAIR]
        """
        B, C4, H, W, D = x.shape
        assert C4 == 4, "انتظار می‌رود کانال ورودی ۴ مدالیتی باشد."

        t1, t1gd, t2, fl = x[:,0:1], x[:,1:2], x[:,2:3], x[:,3:4]  # (B,1,H,W,D)

        print(' -------- preprocess ---------')

        # --- Encoder ---
        enc = self.encoder(t1, t1gd, t2, fl)
        # enc[mod] = [L2@1/4, L3@1/8, L4@1/16]  هر کدام: (B, C, D_l, H_l, W_l)
        t1_L2, t1_L3, t1_L4       = enc['t1']
        t1gd_L2, t1gd_L3, t1gd_L4 = enc['t1gd']
        t2_L2, t2_L3, t2_L4       = enc['t2']
        fl_L2, fl_L3, fl_L4       = enc['flair']

        print(' -------- Encoder ---------')

        # --- Bottleneck (1/16) ---
        # خروجی: fused@1/16
        fused_1_16, _, _ = self.bottleneck(t1_L4, t1gd_L4, t2_L4, fl_L4)  # (B, C, 1/16)

        print(' -------- Bottleneck ---------')

        # --- Skip fusions ---
        # 1/8: concat 4 modality @L3 → 1×1×1 → 64ch
        skip_1_8_cat = torch.cat([t1_L3, t1gd_L3, t2_L3, fl_L3], dim=1)   # (B, 4C, 1/8)
        skip_1_8 = self.skip1_proj(skip_1_8_cat)                          # (B, 64, 1/8)

        # 1/4: concat 4 modality @L2 → 1×1×1 → 32ch
        skip_1_4_cat = torch.cat([t1_L2, t1gd_L2, t2_L2, fl_L2], dim=1)   # (B, 4C, 1/4)
        skip_1_4 = self.skip2_proj(skip_1_4_cat)

        print(' -------- Skip fusions ---------')                         # (B, 32, 1/4)

        # --- Decoder ---
        y_1x, y_1_8, y_1_4 = self.decoder(fused_1_16, skip_1_8, skip_1_4)  # y_1x @ 1×

        print(' -------- Decoder ---------')

        # --- Heads ---
        logits = self.seg_head(y_1x)  # (B, num_classes, H, W, D)

        print(' -------- Heads ---------')

        if self.deep_supervision and self.training:
            # upsample aux logits به فول‌رز برای loss
            aux1 = self.aux1(y_1_8, full_size=(D, H, W))  # توجه: ترتیب (D,H,W) برای interpolate
            aux2 = self.aux2(y_1_4, full_size=(D, H, W))
            return {"logits": logits, "aux1": aux1, "aux2": aux2}

        print(' -------- deep_supervision ---------')

        return {"logits": logits}


In [ ]:
'''
with torch.no_grad():
    x = torch.randn(1, 4, 128, 128, 128) # یا cpu
    model = MultiModalSegNet(base_ch=32, num_heads=4, num_classes=4,
                             up_method="interp", deep_supervision=True)
    model.train()
    out = model(x)
    print(out["logits"].shape)  # (1,4,224,224,160)
    print(out["aux1"].shape)    # (1,4,160,224,224)
    print(out["aux2"].shape)    # (1,4,160,224,224)

'''

torch.Size([1, 4, 128, 128, 128])
torch.Size([1, 4, 128, 128, 128])
torch.Size([1, 4, 128, 128, 128])


In [ ]:
'''
with torch.no_grad():
    x = torch.randn(1, 4, 224, 224, 160) # یا cpu
    model = MultiModalSegNet(
    base_ch=32,
    num_heads=4,
    num_classes=4,
    deep_supervision=True,
    # مقدار سراسری
    dropout=0.10,
    # overrideهای دلخواه:
    attn_dropout=0.05,   # برای attention (self/cross)
    proj_dropout=0.05,   # برای پروجکشن خروجی attention
    ffn_dropout=0.05,    # برای FFNهای 1x1 داخل SelfModal
    dec_dropout=0.10,    # برای دیکودر/TCFC
    )
    model.train()
    out = model(x)
    print(out["logits"].shape)  # (1,4,224,224,160)
    print(out["aux1"].shape)    # (1,4,160,224,224)
    print(out["aux2"].shape)    # (1,4,160,224,224)
'''

'\nwith torch.no_grad():\n    x = torch.randn(1, 4, 224, 224, 160) # یا cpu\n    model = MultiModalSegNet(\n    base_ch=32,\n    num_heads=4,\n    num_classes=4,\n    deep_supervision=True,\n    # مقدار سراسری\n    dropout=0.10,\n    # overrideهای دلخواه:\n    attn_dropout=0.05,   # برای attention (self/cross)\n    proj_dropout=0.05,   # برای پروجکشن خروجی attention\n    ffn_dropout=0.05,    # برای FFNهای 1x1 داخل SelfModal\n    dec_dropout=0.10,    # برای دیکودر/TCFC\n    )\n    model.train()\n    out = model(x)\n    print(out["logits"].shape)  # (1,4,224,224,160)\n    print(out["aux1"].shape)    # (1,4,160,224,224)\n    print(out["aux2"].shape)    # (1,4,160,224,224)\n'

In [ ]:
'''
import torch
import torch.nn as nn

def list_dropouts(model: nn.Module):
    rows = []
    for name, m in model.named_modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)):
            rows.append((name, type(m).__name__, m.p))
    if not rows:
        print("⚠️ هیچ Dropout* پیدا نشد.")
    else:
        print("✅ لایه‌های Dropout پیدا شدند (name, type, p):")
        for r in rows:
            print("  -", r)

# مثال استفاده:
# model = MultiModalSegNet(... با dropoutهایی که تنظیم کردی ...)
list_dropouts(model)

# اگر در کلاس اصلی مقادیر سراسری را نگه می‌داری:
try:
    print("\n[Global dropout wiring]")
    print("attn_dropout =", model.attn_dropout)
    print("proj_dropout =", model.proj_dropout)
    print("ffn_dropout  =", model.ffn_dropout)
    print("dec_dropout  =", model.dec_dropout)
except AttributeError:
    print("نشد مقادیر سراسری را بخوانم (شاید هنوز اضافه نشده‌اند).")

'''

'\nimport torch\nimport torch.nn as nn\n\ndef list_dropouts(model: nn.Module):\n    rows = []\n    for name, m in model.named_modules():\n        if isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)):\n            rows.append((name, type(m).__name__, m.p))\n    if not rows:\n        print("⚠️ هیچ Dropout* پیدا نشد.")\n    else:\n        print("✅ لایه\u200cهای Dropout پیدا شدند (name, type, p):")\n        for r in rows:\n            print("  -", r)\n\n# مثال استفاده:\n# model = MultiModalSegNet(... با dropoutهایی که تنظیم کردی ...)\nlist_dropouts(model)\n\n# اگر در کلاس اصلی مقادیر سراسری را نگه می\u200cداری:\ntry:\n    print("\n[Global dropout wiring]")\n    print("attn_dropout =", model.attn_dropout)\n    print("proj_dropout =", model.proj_dropout)\n    print("ffn_dropout  =", model.ffn_dropout)\n    print("dec_dropout  =", model.dec_dropout)\nexcept AttributeError:\n    print("نشد مقادیر سراسری را بخوانم (شاید هنوز اضافه نشده\u200cاند).")\n\n'

In [ ]:
'''
# ==== Test 2: behavioral check for dropout (train !=, eval ==) ====
import torch

# ورودی تصادفی ۴-مدالیتی با ابعاد امن (در صورت نیاز H/W/D را بزرگ‌تر کن)
def make_dummy_x(model, B=1, H=32, W=32, D=32):
    device = next(model.parameters()).device
    return torch.randn(B, 4, H, W, D, device=device)

@torch.no_grad()
def first_tensor(obj):
    if isinstance(obj, torch.Tensor):
        return obj
    if isinstance(obj, (list, tuple)) and len(obj) > 0:
        return first_tensor(obj[0])
    if isinstance(obj, dict) and len(obj) > 0:
        return first_tensor(next(iter(obj.values())))
    raise ValueError("خروجی مدل ناشناخته است؛ نتونستم تنسور اصلی رو استخراج کنم.")

@torch.no_grad()
def forward_two_passes_same_input(model, x):
    y1 = model(x)
    y2 = model(x)
    t1 = first_tensor(y1).float().reshape(-1)
    t2 = first_tensor(y2).float().reshape(-1)
    return torch.mean(torch.abs(t1 - t2)).item()

x = make_dummy_x(model, B=1, H=32, W=32, D=32)

# حالت train: باید اختلاف > 0 باشه
model.train()
diff_train = forward_two_passes_same_input(model, x)
print(f"[TRAIN] mean|Δ| = {diff_train:.6f}  (باید > 0 باشه)")

# حالت eval: باید عملاً صفر باشه
model.eval()
diff_eval = forward_two_passes_same_input(model, x)
print(f"[EVAL ] mean|Δ| = {diff_eval:.12f}  (باید ≈ 0 باشه)")

# جمع‌بندی
ok_train = diff_train > 1e-6
ok_eval  = diff_eval  < 1e-7
if ok_train and ok_eval:
    print("✅ Dropout درست کار می‌کند: در train خروجی‌ها متفاوت، در eval یکسان.")
else:
    print("⚠️ نتیجه غیرمنتظره. اگر TRAIN≈0 یا EVAL بزرگ بود، wiring یا pها را بررسی کن.")

'''

'\n# ==== Test 2: behavioral check for dropout (train !=, eval ==) ====\nimport torch\n\n# ورودی تصادفی ۴-مدالیتی با ابعاد امن (در صورت نیاز H/W/D را بزرگ\u200cتر کن)\ndef make_dummy_x(model, B=1, H=32, W=32, D=32):\n    device = next(model.parameters()).device\n    return torch.randn(B, 4, H, W, D, device=device)\n\n@torch.no_grad()\ndef first_tensor(obj):\n    if isinstance(obj, torch.Tensor):\n        return obj\n    if isinstance(obj, (list, tuple)) and len(obj) > 0:\n        return first_tensor(obj[0])\n    if isinstance(obj, dict) and len(obj) > 0:\n        return first_tensor(next(iter(obj.values())))\n    raise ValueError("خروجی مدل ناشناخته است؛ نتونستم تنسور اصلی رو استخراج کنم.")\n\n@torch.no_grad()\ndef forward_two_passes_same_input(model, x):\n    y1 = model(x)\n    y2 = model(x)\n    t1 = first_tensor(y1).float().reshape(-1)\n    t2 = first_tensor(y2).float().reshape(-1)\n    return torch.mean(torch.abs(t1 - t2)).item()\n\nx = make_dummy_x(model, B=1, H=32, W=32, D=32)\n\

In [ ]:
'''
# ==== Test 3: hook-based probe of Dropout layers ====
import torch
import torch.nn as nn

class DropoutProbe:
    def __init__(self):
        self.stats = []
        self.handles = []

    def attach(self, model: nn.Module):
        for name, m in model.named_modules():
            if isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)):
                def hook(mod, inp, out, _name=name, _p=m.p):
                    with torch.no_grad():
                        # توجه: برای Dropout3d، صفرشدن کل نقشه‌های ویژگی هم ممکنه؛
                        # اینجا صرفاً نسبت عناصر صفر رو گزارش می‌کنیم.
                        z = (out == 0).float().mean().item()
                    self.stats.append((_name, _p, z))
                h = m.register_forward_hook(hook)
                self.handles.append(h)

    def detach(self):
        for h in self.handles:
            h.remove()
        self.handles.clear()

    def report(self):
        if not self.stats:
            print("هیچ آماری ثبت نشد (آیا forward اجرا شد؟)")
            return
        print("نام لایه".ljust(56), "|  p تنظیم‌شده | نسبت صفر مشاهده‌شده")
        for name, p, z in self.stats:
            print(f"{name:56s} |     {p:>5.2f}     |       {z:>6.3f}")

# اجرای پروب
probe = DropoutProbe()
probe.attach(model)

model.train()  # باید در train باشه تا dropout فعال بشه
with torch.no_grad():
    _ = model(torch.randn(1, 4, 32, 32, 32, device=next(model.parameters()).device))

probe.detach()
probe.report()

'''

# نکتهٔ تفسیر:
# برای هر لایه‌ای که p=0.10 تنظیم کردی، "نسبت صفر" باید حدود 0.10 ± کمی نوسان باشه.
# اگر 0.00 دیدی در حالی که انتظار >0 داشتی:
#   1) آن مسیر در این پاس فعال نشده بود (اندازه/شاخه/گیت)، یا
#   2) p آن لایه هنوز 0 ست (پارامتر پاس نشده)، یا
#   3) مدل ناخواسته eval بوده. قبل از forward حتماً model.train() را صدا بزن.


'\n# ==== Test 3: hook-based probe of Dropout layers ====\nimport torch\nimport torch.nn as nn\n\nclass DropoutProbe:\n    def __init__(self):\n        self.stats = []\n        self.handles = []\n\n    def attach(self, model: nn.Module):\n        for name, m in model.named_modules():\n            if isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)):\n                def hook(mod, inp, out, _name=name, _p=m.p):\n                    with torch.no_grad():\n                        # توجه: برای Dropout3d، صفرشدن کل نقشه\u200cهای ویژگی هم ممکنه؛\n                        # اینجا صرفاً نسبت عناصر صفر رو گزارش می\u200cکنیم.\n                        z = (out == 0).float().mean().item()\n                    self.stats.append((_name, _p, z))\n                h = m.register_forward_hook(hook)\n                self.handles.append(h)\n\n    def detach(self):\n        for h in self.handles:\n            h.remove()\n        self.handles.clear()\n\n    def report(self):\n        if not self.stats:

In [ ]:
'''
اگر همین را اجرا کردی و ابعاد OK بود، قدم بعدی: Loss & Training loop (Dice+CE)، Scheduler، AMP، و log/metrics؛ هر زمان آماده بودی بگو تا ستاپ تمرین را هم تمیز بچینیم.
'''

'\nاگر همین را اجرا کردی و ابعاد OK بود، قدم بعدی: Loss & Training loop (Dice+CE)، Scheduler، AMP، و log/metrics؛ هر زمان آماده بودی بگو تا ستاپ تمرین را هم تمیز بچینیم.\n'

In [ ]:
#@title عنوانِ این سلول
#@markdown (اختیاری) توضیح کوتاه اینجاست

'''
1) اگر با خطای کمبود حافظه (OOM) روبه‌رو شدی

        tcfc_depthwise=True: کانولوشن‌های محوری داخل TCFC رو Depthwise کن تا خیلی ارزون‌تر بشن.
        کانال‌های دیکودر رو کم کن: مثلاً به‌جای 64→32→16→16 برو روی 48→24→16→16 یا حتی کمتر.
        روش آپ‌سمپل: روی up_method="interp" (interpolate سه‌خطی) بمون؛ از ConvTranspose3d سنگین‌تره.
        AMP روشن: با torch.cuda.amp.autocast و GradScaler آموزش رو FP16 کن تا رم کم‌تری مصرف بشه.

2) وقتی کانال‌های انکدر در سطوح مختلف برابر نیستند

        ما اسکیپ‌ها رو روی هر سطح چهار مدالیتی به‌هم concat می‌کنیم و بعد با 1×1×1 می‌بریم به کانال هدف دیکودر.
        ورودی پروجکشن باید جمعِ کانال‌های چهار مدالیتی همون سطح باشه:
        مثال @L3 (≈1/8): اگر T1=48، T1Gd=64، T2=64، FLAIR=32 → جمع = 208. پس skip1_proj باید in_channels=208 باشه (و مثلاً out=64).
        مثال @L2 (≈1/4): اگر هرکدوم 32 باشن → جمع = 128. پس skip2_proj باید in_channels=128 (و مثلاً out=32).

3) اگر خواستی اسکیپِ 1/2 هم اضافه کنی

        مثل استیج‌های 1/8 و 1/4 عمل کن:
        Upsample → SkipGate + SkipProj + TCFC با اسکیپِ 1/2 → DecoderBlock.
        برایش یک پروجکشن جدا هم بگذار (اسمش هرچی دوست داری؛ مثلاً skip2_5_proj) که ورودی‌اش جمع کانال‌های ۴ مدالیتیِ سطح 1/2 باشه و خروجی‌اش با کانال دیکودر توی همون استیج یکی.

4) Deep Supervision (شاخه‌های کمکی)

        از خروجی‌های میانی (مثلاً @1/8 و @1/4) هم logits بگیر و به فول‌رز upsample کن و در لاست وارد کن:

        وزن‌ها پیشنهادی:
            اصلی (فول‌رز): 1.0
            @1/4: 0.3
            @1/8: 0.2

فقط در آموزش از این شاخه‌ها استفاده کن. موقع ارزیابی (model.eval())، فقط خروجی اصلی رو استفاده کن.

اگر بخوای، با اعداد کانال‌های دقیقت (واقعی دیتاستت) همین ۳ نقطهٔ بالا رو برات شخصی‌سازی می‌کنم تا مطمئن شی هیچ‌جا mismatch کانال یا مصرف رم اضافه نداریم.
'''

'\n1) اگر با خطای کمبود حافظه (OOM) روبه\u200cرو شدی\n\n        tcfc_depthwise=True: کانولوشن\u200cهای محوری داخل TCFC رو Depthwise کن تا خیلی ارزون\u200cتر بشن.\n        کانال\u200cهای دیکودر رو کم کن: مثلاً به\u200cجای 64→32→16→16 برو روی 48→24→16→16 یا حتی کمتر.\n        روش آپ\u200cسمپل: روی up_method="interp" (interpolate سه\u200cخطی) بمون؛ از ConvTranspose3d سنگین\u200cتره.\n        AMP روشن: با torch.cuda.amp.autocast و GradScaler آموزش رو FP16 کن تا رم کم\u200cتری مصرف بشه.\n\n2) وقتی کانال\u200cهای انکدر در سطوح مختلف برابر نیستند\n\n        ما اسکیپ\u200cها رو روی هر سطح چهار مدالیتی به\u200cهم concat می\u200cکنیم و بعد با 1×1×1 می\u200cبریم به کانال هدف دیکودر.\n        ورودی پروجکشن باید جمعِ کانال\u200cهای چهار مدالیتی همون سطح باشه:\n        مثال @L3 (≈1/8): اگر T1=48، T1Gd=64، T2=64، FLAIR=32 → جمع = 208. پس skip1_proj باید in_channels=208 باشه (و مثلاً out=64).\n        مثال @L2 (≈1/4): اگر هرکدوم 32 باشن → جمع = 128. پس skip2_proj باید in_channels=128 (و مثلاً out=32).